# Business Entity Resolution — Stage 1: High-Recall Candidate Generator (SC-Block style)
### From checkpoint: loads your trained `.pt` (no training). Profile: `intfloat/multilingual-e5-small` (MIT), whole notebook ≈ 2 h on Kaggle T4 x2
Time budget: ~25 min training, ~25-35 min validation encoding, ~30-40 min test encoding, ~20-30 min loading/search/writing. The runtime benchmark in Section 14 prints a projection and warns if it exceeds `target_total_minutes`.

> **Core idea.** Fine-tune a supervised contrastive encoder so that S1/S2/S3 records representing the same
> business are close in embedding space, then use FAISS nearest-neighbour search **separately over S2 and S3**
> to generate a high-recall candidate set for every S1 entity.

This notebook **stops at candidate generation**. It outputs `candidate_pairs.tsv` for the test set, along with the
validation metrics needed to decide whether blocking is good enough. It does not build the matcher.

### Why a blocker and not a matcher?
The pairwise space is ~2.2M × 10.3M ≈ 2.3·10¹³ pairs, far too many to score. The blocker only has to make sure the
true matches are **inside** a small candidate set (*recall*). Deciding which candidates really match (*precision*,
which drives F0.5) is the job of the next-stage matcher. Blocking recall is therefore the **upper bound** on final
recall: a true match lost here can never be recovered.

### Pipeline
```
S1 / S2 / S3 ──► conservative normalisation ──► "name: … | address: … | country: …"
                                  │
                SupCon fine-tuned encoder (source-aware batches, SC-Block §3.2–3.3)
                                  │
          ┌───────────────────────┴───────────────────────┐
   FAISS-S2 (per country)                          FAISS-S3 (per country)
   top-K_S2 (or similarity floor)                  top-K_S3 (or similarity floor)
          └───────────────┬───────────────────────────────┘
                          ∪  exact normalised-key matches (only if they add validated recall)
                          ▼
            learned pruner (cheap features, threshold for fewest candidates at target recall)
                          ▼
             candidate_pairs.tsv  (+ test_candidate_scores.parquet for the matcher)
```

### Adaptations and improvements over the plain SC-Block recipe
| # | Change | Why |
|---|---|---|
| 1 | **Consistent DEBUG subsample**: sampled S1 entities + *all* their true matches + random distractors | A random S2/S3 sample would contain almost no true matches, so recall numbers would be meaningless |
| 2 | **Connected components** (scipy) for entity labels; split train/val at the **component** level | Cannot leak a validation group into training, even if two S1s share an S2 record |
| 3 | **Source-aware tables** S1↔S2 and S1↔S3 (paper §3.2) + extra random S1 records as in-batch negatives | S1 is deduplicated, so other S1 records (incl. singletons) are guaranteed negatives |
| 4 | **Per-country FAISS partitioning** (auto-disabled if ground truth contains cross-country matches) | Smaller indexes, faster exact search, no cross-country candidates; France gets its own partition |
| 5 | **GPU exact search (fp16 FlatIP)** when faiss-gpu is available, IVF on CPU otherwise | Exact search avoids IVF recall loss; Kaggle T4s make brute force over 3M vectors cheap |
| 6 | **Search once at K_max, compute ranks, derive every K** | The K sweep, per-source K and threshold sweep cost almost nothing |
| 7 | **Per-source K** (K_S2 ≠ K_S3) chosen as the smallest K reaching the target recall | S2 has ≤5 matches per S1, S3 ≤6; they behave differently |
| 8 | **Dynamic-K (similarity floor)** evaluated against flat top-K; used only if it wins on validation | Can shrink candidate sets for easy entities |
| 9 | **Exact normalised-key union**, kept only if it adds validated recall | Free recall safety net for exact duplicates |
| 10 | **Dedupe-before-encode** + length-sorted batching | Duplicate texts are encoded once; less padding means faster encoding |
| 11 | **Mini-benchmark** (same queries/index for all methods) for TF-IDF vs pretrained vs fine-tuned + serialisation ablation A–D | Answers "does fine-tuning help?" without encoding 10M records four times |
| 12 | **Runtime benchmark + time-budgeted training + resumable checkpoints** | Stays inside Kaggle session limits |
| 13 | **Unseen-country K boost** (France): larger flat K, no similarity floor | We cannot validate France, so we are conservative there |
| 14 | **Tokenizer [UNK] diagnostic per country** | The data contains Devanagari names; an English-only vocabulary may not cover them (see §12) |
| 15 | **Failure tagging** (missing address, script mismatch, same-name-different-address, low name overlap) | Shows what to fix next |
| 16 | Optional **hard-negative mining** phase, adopted only if it improves validation | Paper-compatible phase 3 without making the first run fragile |
| 17 | **Learned candidate pruner** (LightGBM on cosine/rank/string features), threshold tuned on held-out validation for the fewest candidates at `prune_target_recall` | Smaller candidate sets rank higher in final evaluation; singletons can get empty lists |

**Next stage (not implemented here):** `candidate_pairs.tsv` → pairwise features → matching model → F0.5-optimised threshold → `matching_results.tsv`.

**Before running:** upload your checkpoint (e.g. `validation_run_full.pt`) as a Kaggle dataset and add it to this notebook. `init_checkpoint="auto"` finds it under `/kaggle/input` (prefers files named `validation_run*`); or set the exact path in the config. Use the **validation-run** checkpoint: it was not trained on validation groups, so validation recall stays honest.

## 2. Configuration
All settings live here. **Run with `DEBUG_MODE = True` first.** That runs the whole pipeline on a small but
*consistent* subsample (S1 entities + their true matches + distractors) in a few minutes. Switch to
`DEBUG_MODE = False` only after a clean debug run.

Kaggle settings: **GPU T4 x2 (or P100)** and **Internet ON** (needed to download the Hugging Face model and `faiss-gpu`).

In [ ]:
import os, gc, sys, json, math, time, random, zlib, subprocess, importlib.util, csv, shutil
from pathlib import Path
import numpy as np

# =====================================================================
# MASTER SWITCH
#   True  -> full pipeline on a consistent subsample (smoke test, minutes)
#   False -> full data (~2.2M / 5.0M / 5.3M train, ~1.7M / 4.9M / 5.1M test)
# =====================================================================
DEBUG_MODE = True
SEED = 42

def _pick(debug_value, full_value):
    return debug_value if DEBUG_MODE else full_value

CONFIG = {
    # ---------------- DEBUG subsample sizes ----------------
    "debug_train_s1_n": 60_000,
    "debug_train_s2_n": 200_000,      # true matches of the sampled S1 + random distractors up to this size
    "debug_train_s3_n": 200_000,
    "debug_test_s1_n": 20_000,        # DEBUG test run is a random subsample -> NOT a submittable file
    "debug_test_s2_n": 200_000,
    "debug_test_s3_n": 200_000,

    # ---------------- encoder ----------------
    # intfloat/multilingual-e5-small: MIT licence, ~118M params (most are the 250k-token multilingual
    # embedding table; the transformer is 12 layers x 384 hidden, so it is fast), 384-dim, mean pooling.
    # E5 expects a "query: " prefix; we use it on every record (symmetric record-to-record matching).
    "model_name": "intfloat/multilingual-e5-small",
    "model_license": "MIT",
    "pooling": "mean",
    "text_prefix": "query: ",
    "max_seq_len": 64,                # records are ~30-40 tokens; 64 keeps encoding fast (check 'truncated' in Sec. 12)
    "gradient_checkpointing": False,

    # ---------------- time budget (whole notebook ~2 h on Kaggle T4 x2) ----------------
    "target_total_minutes": 120,      # used only to print a warning after the runtime benchmark

    # ---------------- split ----------------
    "val_frac": 0.20,                 # fraction of S1 *components* held out for honest recall estimates

    # ---------------- contrastive training ----------------
    "labels_per_batch": _pick(48, 128),        # entity groups per batch (x ~3-5 records each)
    "max_other_per_label": 4,                  # max S2 (or S3) records per group in a batch
    "extra_s1_negatives_per_batch": _pick(16, 64),  # random S1 records = guaranteed negatives (S1 is deduplicated)
    "epochs": 1,
    "max_train_minutes": _pick(8, 25),         # hard time budget; stops cleanly and checkpoints
    "lr": 5e-5,                                # short schedule on a small model -> higher LR
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "temperature": 0.07,                       # SC-Block Eq. 1
    "grad_clip": 1.0,
    "use_amp": True,                           # fp16 autocast on GPU; the loss is computed in fp32
    "checkpoint_every_steps": 500,
    "resume_from_checkpoint": True,
    "init_checkpoint": "/kaggle/input/datasets/sathishadithiyaasv/amazon-ml-challenge-deleetcode/validation_run_full.pt",                   # path to a trained encoder .pt, or "auto" to search /kaggle/input
    "checkpoint_name_hint": None,              # with "auto": prefer .pt files whose name contains this
    "checkpoint_exclude_hints": ["baai", "bge"],   # never pick the bge-m3 checkpoint for this e5 notebook
    "skip_training": True,                      # True: use init_checkpoint weights as-is (no validation-run training)

    # ---------------- encoding ----------------
    "encode_batch_size": 1024,
    "keep_val_embeddings": False,
    "emb_memmap_dir": None,                    # e.g. "/tmp/er_embeddings" to keep large embeddings out of RAM

    # ---------------- retrieval ----------------
    "partition_by_country": True,              # auto-disabled if GT has > tolerance cross-country pairs
    "cross_country_tolerance": 0.001,
    "faiss_use_gpu": True,
    "faiss_cpu_ivf_threshold": 1_000_000,      # CPU only: exact FlatIP below, IVFFlat above
    "faiss_nprobe": 32,
    "search_query_chunk": 32_768,
    "k_max": 100,                              # retrieve once at K_max per source, derive all K from ranks
    "pool_k_cap": 30,                          # max stage-1 K per source (RAM + candidate size); pruner shrinks further
    "k_sweep": [1, 2, 3, 5, 10, 20, 50, 100],
    "target_recall": 0.995,                    # SC-Block tunes K for 99.5% validation recall; revisit after the sweep
    "k_safety_factor": 1.0,                    # >1.0 inflates the chosen K for val->test drift
    "threshold_grid": [0.70, 0.75, 0.80, 0.85, 0.90],   # E5 cosines are compressed into a high range
    "dynamic_k_min": 2,                        # dynamic mode always keeps the top-k_min regardless of similarity
    "unseen_country_k_multiplier": 2.0,        # e.g. France: K x 2, never a similarity floor

    # ---------------- exact-key union ----------------
    "exact_full_key_cap": 50,
    "exact_name_key_cap": 10,
    "union_min_recall_gain": 1e-4,

    # ---------------- candidate pruner (final, smaller candidate set) ----------------
    "use_pruner": True,
    "prune_target_recall": 0.985,              # FINAL pair recall kept after pruning (0.99 -> 0.985 cut candidates 4x on e5 full run)
    "prune_unseen_target_recall": 0.988,       # safer threshold for countries unseen in training (France)
    "export_val_candidates": True,             # Section 23c: labelled validation candidates for the matcher
    "prune_fallback_retention": 0.998,         # if the target is above pool recall: keep this share of pool recall
    "prune_n_queries": _pick(10_000, 100_000), # validation queries used (half fit, half threshold selection)

    # ---------------- baselines / ablation (mini-benchmark) ----------------
    "mini_n_queries": _pick(5_000, 10_000),
    "mini_n_distractors": _pick(50_000, 100_000),   # per source, on top of the true matches
    "mini_k_list": [1, 5, 10, 20, 50],
    "run_tfidf_baseline": True,
    "tfidf_ngram_range": (2, 4),
    "run_ablation": _pick(True, False),        # debug only, to stay inside the 2 h budget
    "run_pretrained_full_baseline": False,     # would re-encode 10M records; off

    # ---------------- OPTIONAL hard negatives (phase 3) ----------------
    "run_hard_negatives": False,               # adds ~40+ min (re-encode + retrain + re-validate)
    "hn_n_anchors": _pick(5_000, 100_000),
    "hn_search_k": 10,
    "hn_per_anchor": 3,
    "hn_epochs": 1,
    "hn_max_train_minutes": _pick(5, 15),

    # ---------------- final encoder ----------------
    "final_train_mode": "reuse",               # no extra training; K / floor exactly calibrated for this model
    "final_epochs": 1,
    "final_max_train_minutes": _pick(5, 15),
    "final_lr_scale": 0.5,

    # ---------------- outputs ----------------
    "n_failure_examples": 8,
    "write_test_scores": True,
    "run_official_validator": _pick(True, False),
}

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else "./working")
OUT_DIR = WORK_DIR / "output"
MODEL_DIR = WORK_DIR / "embedding_model"
CKPT_DIR = WORK_DIR / "checkpoints"
for _d in (WORK_DIR, OUT_DIR, MODEL_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

T_NOTEBOOK_START = time.time()
print(f"DEBUG_MODE = {DEBUG_MODE} | running on Kaggle: {IS_KAGGLE} | work dir: {WORK_DIR.resolve()}")
print(json.dumps(CONFIG, indent=1, default=str))

## 3. Kaggle dataset discovery
Kaggle mounts datasets at `/kaggle/input/<dataset-name>/...` and the folder name varies. Set `DATA_ROOT` to the
folder that contains `train/` and `test/` if you know it. Otherwise (or if it is wrong) we search for
`train_source1.tsv`. `test_source3.tsv` is looked up in both `test/` and the dataset root, because the spec shows both
layouts.

In [ ]:
DATA_ROOT = None   # e.g. "/kaggle/input/<dataset>/student_resource/dataset"; None -> auto-discover

def _is_data_root(p):
    return p is not None and os.path.isfile(os.path.join(p, "train", "train_source1.tsv"))

def discover_data_root(explicit=None):
    if _is_data_root(explicit):
        return os.path.abspath(explicit)
    if explicit:
        print(f"DATA_ROOT={explicit!r} does not contain train/train_source1.tsv -> searching")
    for root in ["/kaggle/input", os.getcwd()]:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if "train_source1.tsv" in filenames and os.path.basename(dirpath) == "train":
                return os.path.abspath(os.path.dirname(dirpath))
    raise FileNotFoundError("train/train_source1.tsv not found under /kaggle/input or the working directory; "
                            "set DATA_ROOT manually.")

def first_existing(paths):
    for p in paths:
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"None of these exist: {paths}")

DATA_ROOT = discover_data_root(DATA_ROOT)
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")
PATHS = {
    "train_s1": os.path.join(TRAIN_DIR, "train_source1.tsv"),
    "train_s2": os.path.join(TRAIN_DIR, "train_source2.tsv"),
    "train_s3": os.path.join(TRAIN_DIR, "train_source3.tsv"),
    "train_gt": os.path.join(TRAIN_DIR, "train_ground_truth.tsv"),
    "test_s1": os.path.join(TEST_DIR, "test_source1.tsv"),
    "test_s2": os.path.join(TEST_DIR, "test_source2.tsv"),
    "test_s3": first_existing([os.path.join(TEST_DIR, "test_source3.tsv"),
                               os.path.join(DATA_ROOT, "test_source3.tsv")]),
}
RESOURCE_ROOT = os.path.dirname(DATA_ROOT)       # student_resource/ (holds utils/)
VALIDATOR_PATH = next((p for p in [os.path.join(RESOURCE_ROOT, "utils", "validate_submission.py"),
                                   os.path.join(DATA_ROOT, "utils", "validate_submission.py")]
                       if os.path.isfile(p)), None)

print(f"Using DATA_ROOT = {DATA_ROOT}")
for k, p in PATHS.items():
    print(f"  {k:9s} {p}  ({os.path.getsize(p)/1e6:,.0f} MB)")
print(f"Official validator: {VALIDATOR_PATH or 'not found (own checks will still run)'}")

## 4. Install / import dependencies
FAISS is probed in a **subprocess** before it is imported, so we can still switch to `faiss-gpu-cu12` if the
preinstalled build is CPU-only. If the GPU wheel cannot be installed, the notebook falls back to CPU FAISS (exact
flat index for small partitions, IVF for large ones).

In [ ]:
def pip_install(*pkgs):
    print("pip install", *pkgs)
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs]).returncode == 0

def faiss_probe():
    """None if faiss cannot be imported, else the number of GPUs faiss sees."""
    r = subprocess.run([sys.executable, "-c",
                        "import faiss; print(faiss.get_num_gpus() if hasattr(faiss, 'get_num_gpus') else 0)"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return None
    try:
        return int(r.stdout.strip().splitlines()[-1])
    except Exception:
        return 0

import torch
HAS_CUDA = torch.cuda.is_available()

_st = faiss_probe()
if HAS_CUDA and CONFIG["faiss_use_gpu"] and not _st:
    if pip_install("faiss-gpu-cu12"):
        _st = faiss_probe()
if _st is None:
    pip_install("faiss-cpu")
    _st = faiss_probe()
assert _st is not None, "FAISS could not be installed."

_rf_ok = subprocess.run([sys.executable, "-c", "from rapidfuzz.process import cpdist"], capture_output=True).returncode == 0
if not _rf_ok:
    pip_install("rapidfuzz>=3.6")
for _mod, _pkg in [("transformers", "transformers"), ("pyarrow", "pyarrow"), ("sklearn", "scikit-learn")]:
    if importlib.util.find_spec(_mod) is None:
        pip_install(_pkg)

import faiss
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
from rapidfuzz import fuzz, process as rf_process

DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
N_GPU = torch.cuda.device_count() if HAS_CUDA else 0
FAISS_GPU = bool(HAS_CUDA and CONFIG["faiss_use_gpu"] and hasattr(faiss, "StandardGpuResources")
                 and faiss.get_num_gpus() > 0)
USE_AMP = bool(CONFIG["use_amp"] and HAS_CUDA)
torch.manual_seed(SEED)

print(f"torch {torch.__version__} | pandas {pd.__version__} | faiss {getattr(faiss, '__version__', '?')}")
print(f"CUDA available: {HAS_CUDA} | GPUs: {N_GPU} | FAISS on GPU: {FAISS_GPU} | AMP: {USE_AMP}")
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB")
if not HAS_CUDA:
    print("WARNING: no GPU - training/encoding will be very slow outside DEBUG_MODE.")

def free_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

def ram_gb():
    try:
        import psutil
        return f"{psutil.Process().memory_info().rss/1e9:.1f} GB RSS"
    except Exception:
        return "n/a"

## 5. Load training data
* Every column is read as `str` (`dtype=str`), so IDs like `S1-00007` never become numbers.
* `keep_default_na=False` keeps empty fields as `""` instead of NaN. Names like "NA" or "None" stay literal strings.
* `quoting=QUOTE_NONE`: the files are plain TSV. With default CSV quoting, a stray `"` in a business name would
  silently swallow tabs and newlines. The row count is checked against the raw line count to catch that.

In [ ]:
SOURCE_COLS = ["entity_id", "business_name", "business_address", "country"]

def count_data_lines(path):
    n, last = 0, b"\n"
    with open(path, "rb") as f:
        while True:
            buf = f.read(1 << 24)
            if not buf:
                break
            n += buf.count(b"\n")
            last = buf[-1:]
    return n + (last != b"\n") - 1          # minus header

def read_tsv(path, expected_cols):
    t = time.time()
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False,
                     quoting=csv.QUOTE_NONE, engine="c")
    missing = [c for c in expected_cols if c not in df.columns]
    assert not missing, f"{path}: missing columns {missing}; got {list(df.columns)}"
    df = df[expected_cols]
    for c in expected_cols:
        df[c] = df[c].fillna("").astype(str).str.strip()
    n_lines = count_data_lines(path)
    flag = "" if n_lines == len(df) else f"  <-- WARNING: file has {n_lines:,} data lines"
    print(f"  {os.path.basename(path)}: {len(df):,} rows in {time.time()-t:.0f}s{flag}")
    return df

def read_source(path):
    return read_tsv(path, SOURCE_COLS)

print("Loading training data ...")
s1 = read_source(PATHS["train_s1"])
s2 = read_source(PATHS["train_s2"])
s3 = read_source(PATHS["train_s3"])
gt_raw = read_tsv(PATHS["train_gt"], ["source1_entity_id", "matched_entity_ids"])
print("RAM:", ram_gb())

## 6. Basic sanity checks
These run on the **full** loaded files (before any DEBUG subsampling) and are compared with your EDA numbers.
A silent parse bug shows up here instead of three hours into training.

In [ ]:
EDA_REFERENCE = {
    "S1": {"rows": 2_206_821, "missing_address": 0},
    "S2": {"rows": 5_034_616, "missing_address": 168_967},
    "S3": {"rows": 5_285_603, "missing_address": 175_916},
    "GT": {"rows": 2_206_821, "empty_matches": 123_247},
}

def sanity_report(name, df):
    rep = {
        "rows": len(df),
        "empty_name": int((df["business_name"] == "").sum()),
        "missing_address": int((df["business_address"] == "").sum()),
        "empty_country": int((df["country"] == "").sum()),
        "dup_entity_id": int(df["entity_id"].duplicated().sum()),
        "dup_name": int(df["business_name"].duplicated().sum()),
        "dup_address": int(df["business_address"].duplicated().sum()),
    }
    print(f"--- {name} ---")
    for k, v in rep.items():
        ref = EDA_REFERENCE.get(name, {}).get(k)
        note = "" if ref is None else ("  (matches EDA)" if ref == v else f"  (EDA: {ref:,})")
        print(f"  {k:16s} {v:>12,}{note}")
    print("  countries:", df["country"].value_counts().to_dict())
    return rep

SANITY = {n: sanity_report(n, d) for n, d in [("S1", s1), ("S2", s2), ("S3", s3)]}

n_empty_gt = int((gt_raw["matched_entity_ids"] == "").sum())
print(f"--- GT ---\n  rows {len(gt_raw):,} | empty match lists (singletons): {n_empty_gt:,} "
      f"({n_empty_gt/len(gt_raw):.2%}) | EDA: {EDA_REFERENCE['GT']['empty_matches']:,}")

for name, df, prefix in [("S1", s1, "S1-"), ("S2", s2, "S2-"), ("S3", s3, "S3-")]:
    bad = int((~df["entity_id"].str.startswith(prefix)).sum())
    assert bad == 0, f"{bad} ids in {name} do not start with {prefix}"
    assert SANITY[name]["dup_entity_id"] == 0, f"duplicate entity_id in {name}"
assert gt_raw["source1_entity_id"].isin(set(s1["entity_id"])).all(), "GT references unknown S1 ids"
print("\nPrefix / uniqueness / GT-coverage checks passed.")

## 7. Parse ground truth (+ consistent DEBUG subsample)
`matched_entity_ids` is expanded into explicit `(S1, S2|S3)` pairs. **One S1 can have many matches, and we never
collapse to a single match.**

**DEBUG subsample (improvement #1).** We sample S1 entities, keep **all** of their true S2/S3 matches, then add random
S2/S3 distractors up to the target size. Recall measured in DEBUG mode is then meaningful, although optimistic,
because the index is ~25× smaller than the real one.

In [ ]:
def explode_gt(gt):
    g = gt[gt["matched_entity_ids"].str.len() > 0]
    p = (g.assign(other_id=g["matched_entity_ids"].str.split(","))
          .explode("other_id")[["source1_entity_id", "other_id"]])
    p["other_id"] = p["other_id"].astype(str).str.strip()
    p = p[p["other_id"] != ""].drop_duplicates(ignore_index=True)
    return p

pairs = explode_gt(gt_raw)
bad_prefix = ~(pairs["other_id"].str.startswith("S2-") | pairs["other_id"].str.startswith("S3-"))
print(f"GT pairs: {len(pairs):,} | non S2/S3 ids in GT: {int(bad_prefix.sum())}")
pairs = pairs[~bad_prefix]

if DEBUG_MODE:
    rng = np.random.default_rng(SEED)
    s1 = s1.iloc[np.sort(rng.choice(len(s1), min(CONFIG["debug_train_s1_n"], len(s1)), replace=False))]
    s1 = s1.reset_index(drop=True)
    pairs = pairs[pairs["source1_entity_id"].isin(set(s1["entity_id"]))]
    gt_raw = gt_raw[gt_raw["source1_entity_id"].isin(set(s1["entity_id"]))]

    def _subsample(df, prefix, target):
        need = df["entity_id"].isin(set(pairs.loc[pairs["other_id"].str.startswith(prefix), "other_id"])).values
        pool = np.flatnonzero(~need)
        extra = rng.choice(pool, min(max(0, target - int(need.sum())), len(pool)), replace=False)
        mask = need.copy()
        mask[extra] = True
        print(f"  {prefix} kept {int(need.sum()):,} true-match records + {len(extra):,} distractors")
        return df[mask].reset_index(drop=True)

    s2 = _subsample(s2, "S2-", CONFIG["debug_train_s2_n"])
    s3 = _subsample(s3, "S3-", CONFIG["debug_train_s3_n"])
    print(f"DEBUG subsample -> S1 {len(s1):,} | S2 {len(s2):,} | S3 {len(s3):,} | pairs {len(pairs):,}")
    free_memory()

n_match = pairs.groupby("source1_entity_id").size().reindex(s1["entity_id"]).fillna(0).astype(int).values
print("\nMatches per S1 (0 = singleton):")
print(pd.Series(n_match).value_counts().sort_index().to_string())
print(f"mean {n_match.mean():.3f} | S2 pairs {int(pairs['other_id'].str.startswith('S2-').sum()):,} "
      f"| S3 pairs {int(pairs['other_id'].str.startswith('S3-').sum()):,}")

## 8. Normalise + serialise records
**Conservative normalisation:** Unicode NFKC → lowercase → punctuation to space → collapse whitespace. We keep
numbers, address tokens and legal suffixes, because the address is what separates two businesses with the same name
(S1 has only 1.54M unique names in 2.2M rows).

The punctuation class is an explicit list (ASCII, Latin-1, general punctuation, CJK, Devanagari danda). A generic
`[^\w\s]` would treat Devanagari vowel signs (matras) as punctuation and break Hindi words apart.

**Serialisation** labels each field (SC-Block §3.1 uses `[COL] attr [VAL] value`; we use a shorter equivalent that
costs fewer tokens):
`name: abc restaurant | address: 123 mg road bengaluru | country: india`

We also compute:
* `country_code`: an integer code over an **open** vocabulary (France and any other new country get a new code)
* `key_full` / `key_name`: 64-bit hashes of the normalised `(name, address, country)` and `(name, country)`,
  used for exact-match candidates

In [ ]:
_PUNCT = r"[!-/:-@\[-`{-~¡-¿‐-‧‰-⁞　-〿।॥]"

def normalize_series(s):
    s = pd.Series(s).fillna("").astype(str)
    s = s.str.normalize("NFKC").str.lower()
    s = s.str.replace(_PUNCT, " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s.reset_index(drop=True)

# Serialisation views for the ablation (Section 20). The PRIMARY model uses view D.
VIEWS = {
    "A_name":                 ("name",),
    "B_address":              ("address",),
    "C_name_address":         ("name", "address"),
    "D_name_address_country": ("name", "address", "country"),
}
PRIMARY_VIEW = "D_name_address_country"

def normalized_fields(df):
    return {"name": normalize_series(df["business_name"]),
            "address": normalize_series(df["business_address"]),
            "country": normalize_series(df["country"])}

def serialize(df, view=PRIMARY_VIEW, norm=None):
    norm = norm if norm is not None else normalized_fields(df)
    out = None
    for f in VIEWS[view]:
        part = f + ": " + norm[f]
        out = part if out is None else out + " | " + part
    return out.to_numpy(dtype=object)

COUNTRY_VOCAB = {}   # normalised country string -> int code (open set)

def country_codes(norm_country):
    uniq_codes, uniq = pd.factorize(norm_country)
    lut = np.array([COUNTRY_VOCAB.setdefault(c, len(COUNTRY_VOCAB)) for c in uniq], dtype=np.int16)
    return lut[uniq_codes]

def country_name(code):
    return {v: k for k, v in COUNTRY_VOCAB.items()}.get(int(code), "?")

def hash64(s):
    return pd.util.hash_pandas_object(pd.Series(s), index=False).to_numpy(dtype=np.uint64)

def prepare_source(df, name):
    t = time.time()
    norm = normalized_fields(df)
    df = df.reset_index(drop=True)
    df["text"] = serialize(df, PRIMARY_VIEW, norm)
    df["country_code"] = country_codes(norm["country"])
    has_name = (norm["name"] != "").to_numpy()
    df["key_full"] = np.where(has_name, hash64(norm["name"] + "||" + norm["address"] + "||" + norm["country"]),
                              0).astype(np.uint64)
    df["key_name"] = np.where(has_name, hash64(norm["name"] + "||" + norm["country"]), 0).astype(np.uint64)
    # fields used by the candidate pruner (Section 23b)
    df["norm_name"] = norm["name"].to_numpy(dtype=object)
    df["norm_address"] = norm["address"].to_numpy(dtype=object)
    df["digits"] = norm["address"].str.findall(r"\d+").str.join(" ").to_numpy(dtype=object)
    df["is_deva"] = df["business_name"].str.contains(r"[\u0900-\u097F]", regex=True).to_numpy()
    n_uniq = pd.Series(df["text"]).nunique()
    print(f"  {name}: {len(df):,} rows, {n_uniq:,} unique serialised texts "
          f"({1 - n_uniq/max(len(df),1):.1%} duplicates) in {time.time()-t:.0f}s")
    return df

print("Normalising + serialising ...")
s1 = prepare_source(s1, "S1")
s2 = prepare_source(s2, "S2")
s3 = prepare_source(s3, "S3")
TRAIN_COUNTRY_CODES = set(np.unique(np.concatenate([s1.country_code, s2.country_code, s3.country_code])).tolist())
print("Country vocabulary:", COUNTRY_VOCAB)
for t in s1["text"].values[:3]:
    print("  ", t)
print("RAM:", ram_gb())

## 9. Construct entity labels (correspondence graph → connected components)
As in SC-Block §3.2, we build a graph whose nodes are records and whose edges are GT matches. Each **connected
component** is one entity label. All S1/S2/S3 records in a component are positives for one another (an S1 with
three S2 matches and two S3 matches gives one group of six records). We use `scipy` connected components, which
are vectorised and fast at 12.5M nodes.

**Singletons:** S1 records with no match are components of size 1. They have no positive, so they are never
anchors with positives. They are still used as in-batch **negatives**, which is safe because S1 is deduplicated.
They are also still embedded and evaluated.

**Cross-country diagnostic:** if (almost) no GT pair crosses countries, retrieval can be partitioned by country
(improvement #4).

In [ ]:
n1, n2, n3 = len(s1), len(s2), len(s3)
OFF2, OFF3 = n1, n1 + n2
N_NODES = n1 + n2 + n3
SRC_BOUNDS = {2: (OFF2, OFF3), 3: (OFF3, N_NODES)}

s1_index = pd.Index(s1["entity_id"])
s2_index = pd.Index(s2["entity_id"])
s3_index = pd.Index(s3["entity_id"])

_q = s1_index.get_indexer(pairs["source1_entity_id"].values)
_oid = pairs["other_id"].values
_is2 = pairs["other_id"].str.startswith("S2-").values
_o = np.full(len(pairs), -1, dtype=np.int64)
_o[_is2] = s2_index.get_indexer(_oid[_is2])
_o[~_is2] = s3_index.get_indexer(_oid[~_is2])
_ok = (_q >= 0) & (_o >= 0)
print(f"GT pairs whose ids exist in the loaded files: {_ok.sum():,} / {len(pairs):,}")

PAIR_Q = _q[_ok].astype(np.int64)                       # S1 row
PAIR_SRC = np.where(_is2[_ok], 2, 3).astype(np.int8)    # 2 = S2, 3 = S3
PAIR_O = _o[_ok].astype(np.int64)                       # row inside its own source frame
PAIR_NODE = np.where(PAIR_SRC == 2, OFF2 + PAIR_O, OFF3 + PAIR_O)

graph = coo_matrix((np.ones(len(PAIR_Q), dtype=np.int8), (PAIR_Q, PAIR_NODE)), shape=(N_NODES, N_NODES))
N_COMP, NODE_COMP = connected_components(graph, directed=False)
NODE_COMP = NODE_COMP.astype(np.int64)
del graph, _q, _o, _oid, _is2, _ok
free_memory()

S1_COMP = NODE_COMP[:n1]
S1_N_MATCH = np.bincount(PAIR_Q, minlength=n1)
S1_IS_SINGLETON = S1_N_MATCH == 0
s1_per_comp = np.bincount(S1_COMP, minlength=N_COMP)
print(f"Components: {N_COMP:,} | components with >1 S1 record: {(s1_per_comp > 1).sum():,} "
      f"(S1 is deduplicated, so expect ~0)")
print(f"S1 singletons: {S1_IS_SINGLETON.sum():,} ({S1_IS_SINGLETON.mean():.2%})")

S1_CC = s1["country_code"].to_numpy()
S2_CC = s2["country_code"].to_numpy()
S3_CC = s3["country_code"].to_numpy()
_other_cc = np.where(PAIR_SRC == 2, S2_CC[np.where(PAIR_SRC == 2, PAIR_O, 0)], S3_CC[np.where(PAIR_SRC == 3, PAIR_O, 0)])
CROSS_COUNTRY_FRAC = float((S1_CC[PAIR_Q] != _other_cc).mean()) if len(PAIR_Q) else 0.0
PARTITION = bool(CONFIG["partition_by_country"] and CROSS_COUNTRY_FRAC <= CONFIG["cross_country_tolerance"])
print(f"Cross-country GT pairs: {CROSS_COUNTRY_FRAC:.4%} -> per-country FAISS partitioning: {PARTITION}")

# All record texts in node order (object array of references; no string copies).
TEXT = np.concatenate([s1["text"].to_numpy(object), s2["text"].to_numpy(object), s3["text"].to_numpy(object)])

## 10. Train / validation split (S1-entity level, before any batch is built)
We split **components** (≈ S1 entities) 80/20, stratified by *match-count bucket × country*. Validation groups never
enter contrastive training, so validation recall is honest. Validation S1 records query the **full** S2/S3 index, which
still contains their true matches plus all other records as distractors, just like at test time.

In [ ]:
bucket = np.select([S1_N_MATCH == 0, S1_N_MATCH <= 2, S1_N_MATCH <= 5], ["0", "1-2", "3-5"], "6+")
strat = pd.Series(bucket).str.cat(s1["country"].values, sep="_")
comp_df = pd.DataFrame({"comp": S1_COMP, "strat": strat.values}).drop_duplicates("comp")
comp_df["u"] = np.random.default_rng(SEED + 1).random(len(comp_df))
comp_df["pct"] = comp_df.groupby("strat")["u"].rank(pct=True)

COMP_SPLIT = np.full(N_COMP, -1, dtype=np.int8)                  # -1: no S1 | 0: train | 1: val
COMP_SPLIT[comp_df["comp"].values] = np.where(comp_df["pct"].values <= CONFIG["val_frac"], 1, 0)
S1_SPLIT = COMP_SPLIT[S1_COMP]
TRAIN_S1_ROWS = np.flatnonzero(S1_SPLIT == 0)
VAL_S1_ROWS = np.flatnonzero(S1_SPLIT == 1)

# Leakage check: every matched record must be in the same split as its S1.
assert (COMP_SPLIT[NODE_COMP[PAIR_NODE]] == S1_SPLIT[PAIR_Q]).all()

split_tbl = pd.crosstab([np.where(S1_SPLIT == 1, "val", "train"), s1["country"].values],
                        np.where(S1_IS_SINGLETON, "singleton", "has_match"), margins=True)
print(split_tbl)
print(f"\nTrain S1: {len(TRAIN_S1_ROWS):,} | Val S1: {len(VAL_S1_ROWS):,} "
      f"(val singleton rate {S1_IS_SINGLETON[VAL_S1_ROWS].mean():.2%})")

## 11. Build the contrastive training dataset (source-aware sampling)
SC-Block §3.2: labels are incomplete *across* sources, so an unlabelled S2 record may still match some S1 record.
Treating every other record as a negative can therefore inject label noise. We build **two source-aware tables**:

* **S1↔S2**: train components that contain an S1 record **and** at least one S2 record. A batch holds the S1 record
  plus up to `max_other_per_label` of its S2 matches.
* **S1↔S3**: the same with S3.

Batches alternate between the tables in shuffled order. Every anchor has at least one positive in the batch, and
the other groups provide in-batch negatives. We also add `extra_s1_negatives_per_batch` random **S1** records,
including singletons. S1 is deduplicated, so two different S1 records are guaranteed non-matches. These cheap,
clean negatives teach the encoder that S1 singletons should *not* land close to other businesses.

We store no pair tables. Components are stored in CSR form (`ORDER`, `COMP_OFFSETS`) and batches are assembled
on the fly.

In [ ]:
ORDER = np.argsort(NODE_COMP, kind="stable")
COMP_OFFSETS = np.concatenate([[0], np.cumsum(np.bincount(NODE_COMP, minlength=N_COMP))])
CNT_S1 = np.bincount(NODE_COMP[:n1], minlength=N_COMP)
CNT_S2 = np.bincount(NODE_COMP[OFF2:OFF3], minlength=N_COMP)
CNT_S3 = np.bincount(NODE_COMP[OFF3:], minlength=N_COMP)

def eligible_components(splits, cnt_other):
    return np.flatnonzero(np.isin(COMP_SPLIT, splits) & (CNT_S1 > 0) & (cnt_other > 0))

TRAIN_TABLES = [("S1<->S2", eligible_components([0], CNT_S2), 2),
                ("S1<->S3", eligible_components([0], CNT_S3), 3)]
for name, comps, _ in TRAIN_TABLES:
    print(f"{name}: {len(comps):,} train entity groups")

def build_batch(comps, src, rng, neg_pool, n_extra_neg, hard_negs=None):
    """Return (texts, labels). Labels are component ids, so hard negatives / extra negatives
    automatically get their *true* label (positive only if genuinely the same entity)."""
    lo, hi = SRC_BOUNDS[src]
    maxo = CONFIG["max_other_per_label"]
    parts = []
    for c in comps:
        mem = ORDER[COMP_OFFSETS[c]:COMP_OFFSETS[c + 1]]
        parts.append(mem[mem < n1])
        om = mem[(mem >= lo) & (mem < hi)]
        if len(om) > maxo:
            om = rng.choice(om, maxo, replace=False)
        parts.append(om)
        if hard_negs is not None:
            hn = hard_negs.get(int(c))
            if hn is not None:
                parts.append(hn)
    if n_extra_neg and len(neg_pool):
        parts.append(rng.choice(neg_pool, min(n_extra_neg, len(neg_pool)), replace=False))
    nodes = np.unique(np.concatenate(parts))
    return TEXT[nodes].tolist(), NODE_COMP[nodes]

_t, _l = build_batch(TRAIN_TABLES[0][1][:3], 2, np.random.default_rng(0), TRAIN_S1_ROWS, 2)
print("\nExample batch (3 groups + 2 extra S1 negatives):")
for t, l in zip(_t, _l):
    print(f"  label={l:<9d} {t}")

## 12. Model setup
**`intfloat/multilingual-e5-small`**: ~118M parameters (far below the 8B cap), **MIT** licence on its Hugging Face model card, multilingual (Hindi/Devanagari and French). Every input gets the `query: ` prefix E5 expects; mean pooling. Re-verify this on the model page before final submission; the notebook prints what
it uses. It is a sentence-embedding model already trained for semantic similarity, which is a strong starting point
for retrieval, and it is small enough to fine-tune and to encode ~22M records (train + test) within Kaggle limits.

Encoder = transformer → **pooling** (`CONFIG["pooling"]`: `cls` for BGE models, `mean` for sentence-transformers) → **L2 normalisation**. With unit vectors, the inner product
*is* cosine similarity: ‖a‖=‖b‖=1 ⇒ a·b = ‖a‖‖b‖cos θ = cos θ. FAISS inner-product search is therefore cosine search.

**Tokenizer coverage check:** MiniLM-L6's vocabulary is English-centric, and the data contains Devanagari names.
The cell below reports the `[UNK]` rate per country. If India shows a high rate, switch `model_name` to one of the
multilingual alternatives listed in the config (and re-verify its licence).

In [ ]:
MODEL_NAME = CONFIG["model_name"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RecordEncoder(nn.Module):
    """record text -> transformer -> mean pool -> L2-normalised embedding."""
    def __init__(self, name_or_path, pooling="mean"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name_or_path)
        self.pooling = pooling

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == "cls":
            emb = h[:, 0]
        else:
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            emb = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
        return F.normalize(emb.float(), p=2, dim=1)

def make_encoder(name_or_path=MODEL_NAME):
    enc = RecordEncoder(name_or_path, "cls" if CONFIG["pooling"] == "auto" else CONFIG["pooling"])
    if CONFIG["gradient_checkpointing"]:
        enc.backbone.gradient_checkpointing_enable()       # trades ~30% speed for much lower activation memory
    enc = enc.to(DEVICE)
    return nn.DataParallel(enc) if N_GPU > 1 else enc

def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

_PREFIX = CONFIG.get("text_prefix", "")

def tokenize(texts):
    texts = [_PREFIX + t for t in texts] if _PREFIX else list(texts)
    enc = tokenizer(texts, padding=True, truncation=True, max_length=CONFIG["max_seq_len"], return_tensors="pt")
    return enc["input_ids"].to(DEVICE, non_blocking=True), enc["attention_mask"].to(DEVICE, non_blocking=True)

def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

encoder = make_encoder()
EMB_DIM = unwrap(encoder).backbone.config.hidden_size
N_PARAMS = sum(p.numel() for p in unwrap(encoder).parameters())
print(f"Model: {MODEL_NAME}")
print(f"Licence: {CONFIG['model_license']} (verify at https://huggingface.co/{MODEL_NAME})")
print(f"Parameters: {N_PARAMS/1e6:.1f}M (<= 8B: {N_PARAMS <= 8e9}) | embedding dim {EMB_DIM} | "
      f"DataParallel over {max(N_GPU,1)} device(s)")

# Tokenizer coverage per country (sample of S1+S2 texts).
_unk = tokenizer.unk_token_id
_rng = np.random.default_rng(3)
for cc in sorted(TRAIN_COUNTRY_CODES):
    rows = np.flatnonzero(np.concatenate([S1_CC, S2_CC]) == cc)
    samp = TEXT[_rng.choice(rows, min(2000, len(rows)), replace=False)].tolist()
    ids = tokenizer(samp, truncation=True, max_length=CONFIG["max_seq_len"])["input_ids"]
    tot = sum(len(x) for x in ids)
    unk = sum(sum(t == _unk for t in x) for x in ids)
    trunc = np.mean([len(x) >= CONFIG["max_seq_len"] for x in ids])
    print(f"  {country_name(cc):8s} [UNK] token rate {unk/max(tot,1):.2%} | mean tokens {tot/len(ids):.1f} "
          f"| truncated {trunc:.2%}")

## 13. Supervised contrastive loss (SC-Block Eq. 1)
$$\mathcal{L}=\sum_i \frac{-1}{|P(i)|}\sum_{p\in P(i)}\log\frac{\exp(z_i\cdot z_p/\tau)}{\sum_{a\neq i}\exp(z_i\cdot z_a/\tau)}$$
Every record in the batch is an anchor. It is pulled toward **all** same-label records (multiple S2/S3 matches
are all positives) and pushed away from every other record. Anchors without an in-batch positive (the extra S1
negatives) contribute only to the denominators. The loss is computed in fp32 for stability under AMP.

In [ ]:
def supcon_loss(z, labels, temperature):
    z = z.float()
    labels = labels.to(z.device)
    n = z.shape[0]
    sim = (z @ z.T) / temperature
    self_mask = torch.eye(n, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(self_mask, -1e9)                     # exclude i from A(i); finite to avoid 0*inf
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos = (labels[:, None] == labels[None, :]) & ~self_mask
    n_pos = pos.sum(1)
    valid = n_pos > 0
    if not valid.any():
        return (z * 0).sum()
    loss = -(log_prob * pos).sum(1)[valid] / n_pos[valid]
    return loss.mean()

# quick self-test: identical positives -> low loss, shuffled labels -> higher loss
_z = F.normalize(torch.randn(8, 16), dim=1).repeat_interleave(2, 0)
_lab = torch.arange(8).repeat_interleave(2)
print(f"loss(correct labels)={supcon_loss(_z, _lab, 0.07):.3f}  "
      f"loss(random labels)={supcon_loss(_z, _lab[torch.randperm(16)], 0.07):.3f}")

## 14. Runtime estimate + training
Before training, we time a few forward/backward passes and an encoding batch, then extrapolate training time and the
encoding time for the full corpora. Training is capped by `max_train_minutes`. It checkpoints every
`checkpoint_every_steps` and **resumes** from the last checkpoint if the kernel restarts (the same batch plan is
regenerated from the seed and the finished steps are skipped).

One epoch = one pass over every train entity group in both source-aware tables.

In [ ]:
def build_plan(tables, labels_per_batch, seed):
    rng = np.random.default_rng(seed)
    plan = []
    for ti, (_, comps, _) in enumerate(tables):
        perm = rng.permutation(comps)
        plan += [(ti, perm[s:s + labels_per_batch]) for s in range(0, len(perm), labels_per_batch)]
    return [plan[i] for i in rng.permutation(len(plan))]

def train_encoder(model, tables, *, epochs, max_minutes, lr, run_name, neg_pool, hard_negs=None):
    L = CONFIG["labels_per_batch"]
    steps_per_epoch = sum(math.ceil(len(t[1]) / L) for t in tables)
    total = epochs * steps_per_epoch
    seed = SEED + zlib.crc32(run_name.encode()) % 10_000
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG["weight_decay"])
    sched = get_linear_schedule_with_warmup(opt, int(total * CONFIG["warmup_ratio"]), total)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    ckpt_path = CKPT_DIR / f"{run_name}_{'debug' if DEBUG_MODE else 'full'}.pt"   # delete to restart from scratch
    done = 0
    if CONFIG["resume_from_checkpoint"] and ckpt_path.exists():
        st = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        unwrap(model).backbone.load_state_dict(st["model"])
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        done = st["step"]
        print(f"Resumed '{run_name}' from step {done}")

    def save_ckpt(step):
        torch.save({"model": unwrap(model).backbone.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "scaler": scaler.state_dict(), "step": step}, ckpt_path)

    print(f"[{run_name}] epochs={epochs} steps/epoch={steps_per_epoch:,} total={total:,} "
          f"budget={max_minutes} min lr={lr:g}")
    rng = np.random.default_rng(seed + done)
    model.train()
    hist, g, t0, stop = [], 0, time.time(), False
    pbar = tqdm(total=total, initial=done, desc=run_name)
    for ep in range(epochs):
        for ti, comps in build_plan(tables, L, seed + ep):
            if g < done:
                g += 1
                continue
            if (time.time() - t0) / 60 > max_minutes:
                print(f"\nTime budget reached at step {g:,}/{total:,} - stopping cleanly.")
                stop = True
                break
            texts, labels = build_batch(comps, tables[ti][2], rng, neg_pool,
                                        CONFIG["extra_s1_negatives_per_batch"], hard_negs)
            ids, mask = tokenize(texts)
            with autocast():
                z = model(ids, mask)
            loss = supcon_loss(z, torch.as_tensor(labels), CONFIG["temperature"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            g += 1
            hist.append(loss.item())
            pbar.update(1)
            if g % 20 == 0:
                pbar.set_postfix(loss=f"{np.mean(hist[-20:]):.4f}", bs=len(texts))
            if g % CONFIG["checkpoint_every_steps"] == 0:
                save_ckpt(g)
        if stop:
            break
    pbar.close()
    save_ckpt(g)
    model.eval()
    print(f"[{run_name}] finished {g:,} steps in {(time.time()-t0)/60:.1f} min")
    return hist

def benchmark(model, tables, n_steps=6, n_encode=8192):
    rng = np.random.default_rng(0)
    model.train()
    times = []
    for i in range(n_steps + 1):
        _, comps, src = tables[i % len(tables)]
        texts, labels = build_batch(rng.choice(comps, min(CONFIG["labels_per_batch"], len(comps)), replace=False),
                                    src, rng, TRAIN_S1_ROWS, CONFIG["extra_s1_negatives_per_batch"])
        if HAS_CUDA: torch.cuda.synchronize()
        t = time.time()
        ids, mask = tokenize(texts)
        with autocast():
            z = model(ids, mask)
        supcon_loss(z, torch.as_tensor(labels), CONFIG["temperature"]).backward()
        model.zero_grad(set_to_none=True)
        if HAS_CUDA: torch.cuda.synchronize()
        if i > 0:
            times.append(time.time() - t)
    model.eval()
    sample = TEXT[rng.choice(len(TEXT), min(n_encode, len(TEXT)), replace=False)]
    t = time.time()
    encode_texts(model, sample, desc="benchmark-encode", verbose=False)
    return float(np.mean(times)), len(sample) / (time.time() - t)

### 15a. Encoding helper (defined here because the benchmark uses it)
**Dedupe-before-encode:** `pd.factorize` finds unique texts. Each is encoded once and the result is broadcast back
to every row that shares it. Unique texts are encoded **longest first** in batches, which minimises padding and
surfaces any OOM immediately. Embeddings are stored as **float16** (5M × 384 × 2 B ≈ 3.9 GB per source instead of
7.7 GB). The effect on cosine similarity is about 1e-3.

In [ ]:
def alloc_embeddings(n):
    """float16 (n, EMB_DIM) array. Large arrays go to an unlinked temp file (np.memmap): the OS pages them in and out,
    so they do not count as hard RAM, and the disk space is released automatically when the array is deleted."""
    d = CONFIG.get("emb_memmap_dir")
    if d and n >= 1_000_000:
        os.makedirs(d, exist_ok=True)
        need, free = n * EMB_DIM * 2, shutil.disk_usage(d).free
        if free > need * 1.1:
            path = os.path.join(d, f"emb_{time.time_ns()}.f16")
            arr = np.memmap(path, dtype=np.float16, mode="w+", shape=(n, EMB_DIM))
            os.remove(path)
            return arr
        print(f"  (only {free/1e9:.1f} GB free in {d}, need {need/1e9:.1f} GB -> keeping embeddings in RAM)")
    return np.empty((n, EMB_DIM), dtype=np.float16)

@torch.no_grad()
def encode_texts(model, texts, desc="encode", batch_size=None, verbose=True):
    batch_size = batch_size or CONFIG["encode_batch_size"]
    codes, uniq = pd.factorize(np.asarray(texts, dtype=object))
    uniq = np.asarray(uniq, dtype=object)
    lens = np.fromiter((len(t) for t in uniq), dtype=np.int32, count=len(uniq))
    order = np.argsort(-lens, kind="stable")
    # Write each unique embedding straight into every row that shares it (no n_unique x dim copy).
    cnt = np.bincount(codes, minlength=len(uniq))
    bnd = np.concatenate([[0], np.cumsum(cnt)])
    rows_sorted = np.argsort(codes, kind="stable")
    out = alloc_embeddings(len(codes))
    model.eval()
    it = range(0, len(uniq), batch_size)
    if verbose:
        it = tqdm(it, desc=f"{desc}: {len(uniq):,} unique / {len(codes):,} rows")
    for s in it:
        idx = order[s:s + batch_size]
        ids, mask = tokenize(uniq[idx].tolist())
        with autocast():
            z = model(ids, mask)
        z = z.float().cpu().numpy().astype(np.float16)
        lens = cnt[idx]
        offs = np.repeat(bnd[idx] - np.concatenate([[0], np.cumsum(lens)[:-1]]), lens) + np.arange(lens.sum())
        out[rows_sorted[offs]] = np.repeat(z, lens, axis=0)
    del rows_sorted
    return out

sec_per_step, enc_per_sec = benchmark(encoder, TRAIN_TABLES)
L = CONFIG["labels_per_batch"]
steps_per_epoch = sum(math.ceil(len(t[1]) / L) for t in TRAIN_TABLES)
planned = CONFIG["epochs"] * steps_per_epoch
print(f"Train step: {sec_per_step:.3f}s | encoding: {enc_per_sec:,.0f} texts/s")
print(f"Validation-run training: {planned:,} steps ~ {planned*sec_per_step/60:.0f} min "
      f"(budget {CONFIG['max_train_minutes']} min -> {min(planned, int(CONFIG['max_train_minutes']*60/sec_per_step)):,} steps)")
_n_uniq_train = pd.Series(TEXT[OFF2:]).nunique() + len(VAL_S1_ROWS)
print(f"Validation encoding (~{_n_uniq_train:,} unique texts): ~{_n_uniq_train/enc_per_sec/60:.0f} min per model")
_n_test = 1_732_544 + 4_887_273 + 5_082_316   # test row counts from the spec files
print(f"Test encoding (~{_n_test:,} rows before dedupe): ~{_n_test/enc_per_sec/60:.0f} min")
_proj = (min(planned * sec_per_step, CONFIG["max_train_minutes"] * 60) + (_n_uniq_train + 0.95 * _n_test) / enc_per_sec) / 60
print(f"Projected training + all encoding (full data): ~{_proj:.0f} min "
      f"(+ ~20-30 min for loading, FAISS search and writing files)")
if CONFIG.get("target_total_minutes") and not DEBUG_MODE and _proj + 25 > CONFIG["target_total_minutes"]:
    print(f"WARNING: projected runtime exceeds target_total_minutes={CONFIG['target_total_minutes']}. "
          "Lower max_train_minutes / max_seq_len, or accept the longer run.")

In [ ]:
def find_checkpoint(spec):
    if spec and spec != "auto":
        assert os.path.isfile(spec), f"init_checkpoint not found: {spec}"
        return spec
    found = sorted(set(glob.glob("/kaggle/input/**/*.pt", recursive=True) + glob.glob(str(CKPT_DIR / "*.pt"))))
    assert found, "No .pt checkpoint found under /kaggle/input (add it as a Kaggle dataset) or set init_checkpoint."
    print("Checkpoints found:", *found, sep="\n  ")
    hint = (CONFIG.get("checkpoint_name_hint") or "").lower()
    if hint:
        hinted = [f for f in found if hint in os.path.basename(f).lower()]
        assert hinted, f"no .pt file name contains checkpoint_name_hint={hint!r}"
        found = hinted
    excl = [x.lower() for x in (CONFIG.get("checkpoint_exclude_hints") or [])]
    if excl:
        found = [f for f in found if not any(x in os.path.basename(f).lower() for x in excl)]
        assert found, f"all .pt files excluded by checkpoint_exclude_hints={excl}"
    preferred = [f for f in found if "validation_run" in os.path.basename(f)] or found
    return max(preferred, key=os.path.getmtime)

def load_encoder_checkpoint(model, path):
    """Accepts this notebook's training checkpoints ({'model': backbone_state_dict, ...}) or a raw state_dict."""
    st = torch.load(path, map_location="cpu", weights_only=False)
    sd = st["model"] if isinstance(st, dict) and "model" in st else st
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    clean = {}
    for k, v in sd.items():
        for pre in ("module.", "backbone."):
            if k.startswith(pre):
                k = k[len(pre):]
        clean[k] = v
    missing, unexpected = unwrap(model).backbone.load_state_dict(clean, strict=False)
    missing = [k for k in missing if not k.endswith("position_ids")]
    print(f"Loaded encoder weights from {path}" + (f" (trained {st['step']:,} steps)" if isinstance(st, dict) and "step" in st else ""))
    assert len(missing) == 0, f"{len(missing)} missing weights (wrong base model?), e.g. {missing[:5]}"
    if unexpected:
        print(f"  ignored {len(unexpected)} unexpected keys, e.g. {unexpected[:3]}")
    name = os.path.basename(path).lower()
    if "final" in name:
        print("WARNING: a 'final' checkpoint was also trained on validation groups -> validation recall will be optimistic.")
    if "debug" in name:
        print("WARNING: this is a DEBUG-mode checkpoint (short training on a subsample).")

if CONFIG.get("init_checkpoint"):
    import glob
    INIT_CKPT = find_checkpoint(CONFIG["init_checkpoint"])
    load_encoder_checkpoint(encoder, INIT_CKPT)

if CONFIG["pooling"] == "auto":
    # The checkpoint does not record its pooling; pick the one under which true matches are most separated
    # from random records (cosine margin on 2,000 train pairs).
    _r = np.random.default_rng(7)
    _tr = np.flatnonzero(S1_SPLIT[PAIR_Q] == 0)
    _i = _r.choice(_tr, min(2000, len(_tr)), replace=False)
    _a, _b = TEXT[PAIR_Q[_i]], TEXT[PAIR_NODE[_i]]
    _n = TEXT[_r.choice(np.arange(OFF2, N_NODES), len(_i), replace=False)]
    margins = {}
    for pooling in ("cls", "mean"):
        unwrap(encoder).pooling = pooling
        ea, eb, en = (encode_texts(encoder, x, verbose=False).astype(np.float32) for x in (_a, _b, _n))
        margins[pooling] = float((ea * eb).sum(1).mean() - (ea * en).sum(1).mean())
    CONFIG["pooling"] = max(margins, key=margins.get)
    unwrap(encoder).pooling = CONFIG["pooling"]
    print(f"Pooling auto-detect (positive - random cosine margin): {margins} -> using '{CONFIG['pooling']}'")

if CONFIG.get("skip_training"):
    assert CONFIG.get("init_checkpoint"), "skip_training=True requires init_checkpoint"
    train_loss_history = []
    encoder.eval()
    print("skip_training=True -> using the loaded encoder as-is.")
else:
    train_loss_history = train_encoder(
        encoder, TRAIN_TABLES, epochs=CONFIG["epochs"], max_minutes=CONFIG["max_train_minutes"],
        lr=CONFIG["lr"], run_name="validation_run", neg_pool=TRAIN_S1_ROWS)

if train_loss_history:
    sm = pd.Series(train_loss_history).rolling(max(1, len(train_loss_history) // 50), min_periods=1).mean()
    plt.figure(figsize=(8, 3)); plt.plot(sm.values); plt.title("SupCon training loss (smoothed)")
    plt.xlabel("step"); plt.ylabel("loss"); plt.tight_layout(); plt.show()

## 15–17. Embedding generation, FAISS indexes, validation retrieval
* **Two separate indexes, S2 and S3** (never one merged index), so recall and K can be analysed and tuned per source.
* **Per-country partitions** (when `PARTITION`): each country's queries are searched only against that country's
  records. A country absent from the index falls back to the full index.
* **Index type:** with faiss-gpu we use exact `FlatIP` stored in fp16, replicated on each GPU (queries are split
  across GPUs), with no approximation loss. On CPU we use exact `FlatIP` below `faiss_cpu_ivf_threshold` vectors and `IVFFlat` (nlist ≈ 4√n,
  nprobe = `faiss_nprobe`) above it. IVF trades a small, measurable recall drop for a large speed-up.
* We search **once** at `K_max` and store `I` (row ids) and `D` (cosine). Every smaller K, per-source K and
  similarity floor is then derived without searching again.
* There is **no S1×S2 matrix**. FAISS works in query chunks of `search_query_chunk`.

In [ ]:
_GPU_RES = None

ADD_CHUNK = 250_000

def _chunks_f32(x, rows):
    for a in range(0, len(rows), ADD_CHUNK):
        c = np.ascontiguousarray(x[rows[a:a + ADD_CHUNK]], dtype=np.float32)
        faiss.normalize_L2(c)                   # idempotent; guarantees inner product == cosine
        yield c

def build_index(x, rows=None):
    """Index x[rows] (added in float32 chunks so RAM never holds a full float32 copy)."""
    global _GPU_RES
    rows = np.arange(len(x)) if rows is None else rows
    n, d = len(rows), x.shape[1]
    if FAISS_GPU:
        cpu = faiss.IndexFlatIP(d)
        if faiss.get_num_gpus() > 1:
            # Replicas (full copy per GPU, queries split across GPUs). Sharding is NOT used because
            # IndexShards only supports a single add() call, and we add in chunks to save RAM.
            co = faiss.GpuMultipleClonerOptions(); co.useFloat16 = True; co.shard = False
            index = faiss.index_cpu_to_all_gpus(cpu, co)
        else:
            _GPU_RES = _GPU_RES or faiss.StandardGpuResources()
            co = faiss.GpuClonerOptions(); co.useFloat16 = True
            index = faiss.index_cpu_to_gpu(_GPU_RES, 0, cpu, co)
        kind = "GPU FlatIP fp16 (exact)"
    elif n <= CONFIG["faiss_cpu_ivf_threshold"]:
        index = faiss.IndexFlatIP(d)
        kind = "CPU FlatIP (exact)"
    else:
        nlist = int(min(65536, max(256, 4 * math.sqrt(n))))
        index = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
        tr_rows = np.sort(np.random.default_rng(0).choice(rows, min(n, 64 * nlist), replace=False))
        index.train(np.concatenate(list(_chunks_f32(x, tr_rows))))
        index.nprobe = CONFIG["faiss_nprobe"]
        kind = f"CPU IVFFlat (nlist={nlist}, nprobe={index.nprobe}, approximate)"
    for c in _chunks_f32(x, rows):
        index.add(c)
    return index, kind

def search_by_country(q_emb, q_cc, x_emb, x_cc, k, desc=""):
    """k: int, or dict {country_code: k}. Returns I (row ids into x, -1 = empty) and D (cosine, -inf = empty),
    sorted by decreasing similarity."""
    k_of = (lambda c: int(k.get(int(c), max(k.values())))) if isinstance(k, dict) else (lambda c: int(k))
    groups = [(c, np.flatnonzero(q_cc == c)) for c in np.unique(q_cc)] if PARTITION else [(None, np.arange(len(q_emb)))]
    kmax = max(k_of(c) if c is not None else (max(k.values()) if isinstance(k, dict) else k) for c, _ in groups)
    I = np.full((len(q_emb), kmax), -1, dtype=np.int32)
    D = np.full((len(q_emb), kmax), -np.inf, dtype=np.float32)
    free_memory()                               # hand cached torch GPU memory to FAISS
    for c, qidx in groups:
        if c is None:
            xidx, kc = np.arange(len(x_emb)), kmax
        else:
            xidx = np.flatnonzero(x_cc == c)
            if len(xidx) == 0:
                print(f"  WARNING: country '{country_name(c)}' absent from {desc} index -> searching full index")
                xidx = np.arange(len(x_emb))
            kc = k_of(c)
        kc = min(kc, len(xidx))
        if kc == 0 or len(qidx) == 0:
            continue
        t = time.time()
        index, kind = build_index(x_emb, xidx)
        for s in range(0, len(qidx), CONFIG["search_query_chunk"]):
            rows = qidx[s:s + CONFIG["search_query_chunk"]]
            q = np.ascontiguousarray(q_emb[rows], dtype=np.float32)
            faiss.normalize_L2(q)
            d, i = index.search(q, kc)
            ok = i >= 0
            I[rows, :kc] = np.where(ok, xidx[np.where(ok, i, 0)], -1)
            D[rows, :kc] = np.where(ok, d, -np.inf)
        cname = "ALL" if c is None else country_name(c)
        print(f"  {desc:>6s} [{cname}] {kind}: {len(xidx):,} vectors, {len(qidx):,} queries, "
              f"k={kc} in {time.time()-t:.0f}s")
        del index
        free_memory()
    return I, D

In [ ]:
BIG_RANK = np.iinfo(np.int32).max

def make_eval_set(q_s1_rows):
    """Ground-truth pairs for a set of query S1 rows, re-indexed to query positions."""
    pos = np.full(n1, -1, dtype=np.int64)
    pos[q_s1_rows] = np.arange(len(q_s1_rows))
    pq = pos[PAIR_Q]
    ev = {"s1_rows": np.asarray(q_s1_rows), "nq": len(q_s1_rows)}
    for src in (2, 3):
        m = (pq >= 0) & (PAIR_SRC == src)
        ev[f"q{src}"], ev[f"o{src}"] = pq[m], PAIR_O[m]
    ev["n_true"] = np.bincount(ev["q2"], minlength=ev["nq"]) + np.bincount(ev["q3"], minlength=ev["nq"])
    return ev

def ranks_of(I, D, q, o, chunk=250_000):
    """Rank (0-based) of index row o in query q's neighbour list (BIG_RANK if not within K_max) + its cosine."""
    r = np.full(len(q), BIG_RANK, dtype=np.int32)
    s = np.full(len(q), -np.inf, dtype=np.float32)
    for a in range(0, len(q), chunk):
        qa, oa = q[a:a + chunk], o[a:a + chunk]
        M = I[qa] == oa[:, None]
        has, p = M.any(1), M.argmax(1)
        r[a:a + chunk] = np.where(has, p, BIG_RANK)
        s[a:a + chunk] = np.where(has, D[qa, p], -np.inf)
    return r, s

def finish_retrieval(I2, D2, I3, D3, ev):
    R = {"I2": I2, "D2": D2, "I3": I3, "D3": D3}
    R["r2"], R["s2"] = ranks_of(I2, D2, ev["q2"], ev["o2"])
    R["r3"], R["s3"] = ranks_of(I3, D3, ev["q3"], ev["o3"])
    return R

def run_validation_retrieval(model, tag="fine-tuned", keep_emb=True):
    t = time.time()
    q_emb = encode_texts(model, TEXT[VAL_S1_ROWS], desc=f"{tag} val-S1")
    e2 = encode_texts(model, s2["text"].values, desc=f"{tag} S2")
    I2, D2 = search_by_country(q_emb, S1_CC[VAL_S1_ROWS], e2, S2_CC, CONFIG["k_max"], desc="S2")
    if not keep_emb:
        del e2; e2 = None; free_memory()
    e3 = encode_texts(model, s3["text"].values, desc=f"{tag} S3")
    I3, D3 = search_by_country(q_emb, S1_CC[VAL_S1_ROWS], e3, S3_CC, CONFIG["k_max"], desc="S3")
    if not keep_emb:
        del e3; e3 = None; free_memory()
    R = finish_retrieval(I2, D2, I3, D3, EV)
    R.update(q_emb=q_emb, e2=e2, e3=e3)
    print(f"[{tag}] validation retrieval done in {(time.time()-t)/60:.1f} min | RAM {ram_gb()}")
    return R

EV = make_eval_set(VAL_S1_ROWS)
print(f"Validation queries: {EV['nq']:,} | true pairs S2 {len(EV['q2']):,} / S3 {len(EV['q3']):,}")
R_VAL = run_validation_retrieval(encoder, "fine-tuned", keep_emb=CONFIG["keep_val_embeddings"])

## 18. Recall@K evaluation
For a candidate configuration `(K_S2, K_S3[, similarity floor])` we compute:

| metric | meaning |
|---|---|
| `pair_recall` | true pairs retrieved / all true pairs (**primary**) |
| `s2_recall`, `s3_recall` | the same per source |
| `mean_s1_recall` | mean of per-S1 recall (non-singletons) |
| `full_recall_pct` | % of non-singleton S1 whose matches were **all** retrieved |
| `avg_cands`, `p95_cands` | candidates per S1 (= matcher workload) |
| `avg_cands_singleton` | candidates for true singletons: every one is a potential false merge for the matcher |
| `pair_precision` | true pairs / candidates (how much filtering is left for the matcher) |
| `reduction_ratio` | 1 − \|C\| / (\|S1\|·(\|S2\|+\|S3\|)) (SC-Block's efficiency metric) |

Everything is vectorised over the stored ranks. There are no Python loops over queries.

In [ ]:
def _kept(rank, sim, k, thr, kmin):
    h = rank < k
    if thr is not None:
        h &= (sim >= thr) | (rank < kmin)
    return h

def _count_kept(I, D, k, thr, kmin):
    k = min(k, I.shape[1])
    v = I[:, :k] >= 0
    if thr is not None:
        v &= (D[:, :k] >= thr) | (np.arange(k) < kmin)[None, :]
    return v.sum(1)

def evaluate(ev, R, k2, k3, thr=None, kmin=0, extra=None, n_index=None, per_query=False):
    """extra: {src: (q, o, rank, sim)} additional (e.g. exact-match) candidate pairs unioned in."""
    nq = ev["nq"]
    n_index = n_index or (n2, n3)
    hits, cands = {}, {}
    for src, k in ((2, k2), (3, k3)):
        h = _kept(R[f"r{src}"], R[f"s{src}"], k, thr, kmin)
        c = _count_kept(R[f"I{src}"], R[f"D{src}"], k, thr, kmin)
        if extra is not None and src in extra:
            eq, eo, er, es = extra[src]
            new = ~_kept(er, es, k, thr, kmin)
            c = c + np.bincount(eq[new], minlength=nq)
            nx = n_index[src - 2]
            h = h | np.isin(ev[f"q{src}"].astype(np.int64) * nx + ev[f"o{src}"], eq.astype(np.int64) * nx + eo)
        hits[src], cands[src] = h, c
    nh = np.bincount(ev["q2"], weights=hits[2], minlength=nq) + np.bincount(ev["q3"], weights=hits[3], minlength=nq)
    nt = ev["n_true"]
    cand = cands[2] + cands[3]
    ns = nt > 0
    n_pairs = len(ev["q2"]) + len(ev["q3"])
    m = {
        "k_s2": k2, "k_s3": k3, "sim_floor": thr,
        "pair_recall": (hits[2].sum() + hits[3].sum()) / max(n_pairs, 1),
        "s2_recall": hits[2].mean() if len(hits[2]) else np.nan,
        "s3_recall": hits[3].mean() if len(hits[3]) else np.nan,
        "mean_s1_recall": float((nh[ns] / nt[ns]).mean()) if ns.any() else np.nan,
        "full_recall_pct": float((nh[ns] == nt[ns]).mean() * 100) if ns.any() else np.nan,
        "avg_cands": float(cand.mean()),
        "avg_cands_s2": float(cands[2].mean()), "avg_cands_s3": float(cands[3].mean()),
        "p95_cands": float(np.percentile(cand, 95)),
        "avg_cands_singleton": float(cand[~ns].mean()) if (~ns).any() else np.nan,
        "pair_precision": float(nh.sum() / max(cand.sum(), 1)),
        "reduction_ratio": 1.0 - cand.sum() / (nq * (n_index[0] + n_index[1])),
    }
    if per_query:
        return m, {"n_hit": nh, "n_cand": cand, "n_cand_s2": cands[2], "n_cand_s3": cands[3]}
    return m

def show(df, cols=None):
    cols = cols or ["k_s2", "k_s3", "sim_floor", "pair_recall", "s2_recall", "s3_recall", "mean_s1_recall",
                    "full_recall_pct", "avg_cands", "p95_cands", "avg_cands_singleton", "pair_precision",
                    "reduction_ratio"]
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 250):
        print(df[[c for c in cols if c in df.columns]].to_string(index=False))

print("Probe @ K=10 per source:")
print(json.dumps({k: (round(v, 5) if isinstance(v, float) else v) for k, v in evaluate(EV, R_VAL, 10, 10).items()},
                 indent=1))

## 19. K comparison (flat top-K) and 19b. dynamic-K (similarity floor)
* **Why K matters:** K sets the recall ceiling *and* the matcher's workload. On the test set, 1.73M S1 × (K_S2+K_S3)
  pairs is what the matcher must score.
* The table uses the same K for both sources. We then pick **K_S2 and K_S3 separately** as the smallest K whose
  per-source recall reaches `target_recall`, using the full rank curve (every K from 1 to K_max, not just the
  sweep grid). If the target is not reached within K_max, K_max is used and a warning is printed. Increase
  `k_max`, or improve the encoder, in that case.
* **19b:** keep a neighbour only if cosine ≥ floor (always keeping the top `dynamic_k_min`), up to K_max. Dynamic
  mode is used on test **only if** it reaches the target with fewer average candidates than flat per-source K.

The choice is made mechanically from these measurements. Review the tables: if 99.5% costs too many candidates,
change `target_recall` and re-run this cell.

In [ ]:
def recall_curve(rank, kmax):
    if len(rank) == 0:
        return np.ones(kmax)
    return np.cumsum(np.bincount(rank[rank < kmax], minlength=kmax)) / len(rank)

def smallest_k(curve, target):
    ok = np.flatnonzero(curve >= target)
    return (int(ok[0]) + 1, True) if len(ok) else (len(curve), False)

def select_candidate_config(R, ev=EV, verbose=True):
    kmax, target = CONFIG["k_max"], CONFIG["target_recall"]
    sweep = pd.DataFrame([evaluate(ev, R, k, k) for k in CONFIG["k_sweep"] if k <= kmax])
    curves = pd.DataFrame({"k": np.arange(1, kmax + 1),
                           "s2_recall": recall_curve(R["r2"], kmax), "s3_recall": recall_curve(R["r3"], kmax)})
    k2, ok2 = smallest_k(curves["s2_recall"].values, target)
    k3, ok3 = smallest_k(curves["s3_recall"].values, target)
    cap = min(kmax, CONFIG.get("pool_k_cap") or kmax)
    k2 = min(cap, math.ceil(k2 * CONFIG["k_safety_factor"]))
    k3 = min(cap, math.ceil(k3 * CONFIG["k_safety_factor"]))
    flat = evaluate(ev, R, k2, k3)
    thr_df = pd.DataFrame([evaluate(ev, R, cap, cap, thr=t, kmin=CONFIG["dynamic_k_min"])
                           for t in CONFIG["threshold_grid"]])
    qual = thr_df[thr_df["pair_recall"] >= target]
    use_dyn = bool(len(qual) and flat["pair_recall"] >= target and
                   qual["avg_cands"].min() < flat["avg_cands"]) or bool(len(qual) and flat["pair_recall"] < target)
    cfg = {"mode": "flat", "k_s2": k2, "k_s3": k3, "sim_floor": None, "k_min": CONFIG["dynamic_k_min"],
           "dyn_k": cap, "target_reached": bool(flat["pair_recall"] >= target)}
    chosen = flat
    if use_dyn:
        best = qual.sort_values("avg_cands").iloc[0]
        cfg.update(mode="dynamic", sim_floor=float(best["sim_floor"]), target_reached=True)
        chosen = best.to_dict()
    if verbose:
        print("=== Flat top-K, same K for S2 and S3 ===")
        show(sweep)
        print(f"\nPer-source K for target recall {target:.3%}: K_S2={k2} (reached={ok2}), K_S3={k3} (reached={ok3})"
              f" | pool cap {cap} (memory / candidate-size limit; the pruner then shrinks the pool)")
        if not (ok2 and ok3):
            print("WARNING: target not reached within K_max for at least one source - consider a larger k_max "
                  "or better encoder (hard negatives, larger/multilingual model).")
        print("\n=== Chosen flat per-source K ===")
        show(pd.DataFrame([flat]))
        print(f"\n=== 19b Dynamic-K: similarity floor (k_min={CONFIG['dynamic_k_min']}, cap={kmax}) ===")
        show(thr_df)
        print(f"\n-> Candidate config: {cfg}")
    return {"cfg": cfg, "sweep": sweep, "curves": curves, "thr": thr_df, "flat": flat, "chosen": chosen}

SEL = select_candidate_config(R_VAL)
CAND_CFG = SEL["cfg"]

SEL["sweep"].to_csv(OUT_DIR / "recall_at_k.csv", index=False)
SEL["curves"].to_csv(OUT_DIR / "recall_curve_per_source.csv", index=False)
SEL["thr"].to_csv(OUT_DIR / "similarity_threshold_sweep.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(SEL["curves"]["k"], SEL["curves"]["s2_recall"], label="S2")
ax[0].plot(SEL["curves"]["k"], SEL["curves"]["s3_recall"], label="S3")
ax[0].axhline(CONFIG["target_recall"], ls="--", c="gray"); ax[0].set_xscale("log")
ax[0].set_xlabel("K per source"); ax[0].set_ylabel("pair recall"); ax[0].legend(); ax[0].set_title("Recall@K")
_top1 = R_VAL["D2"][:, 0]
for _mask, _lab in ((EV["n_true"] == 0, "singleton S1"), (EV["n_true"] > 0, "S1 with matches")):
    _v = _top1[_mask & np.isfinite(_top1)]
    if len(_v):
        ax[1].hist(_v, bins=50, alpha=0.6, density=True, label=_lab)
ax[1].set_title("Top-1 S2 cosine"); ax[1].legend(); plt.tight_layout(); plt.show()

## 20. Baselines: TF-IDF vs pretrained vs fine-tuned (+ serialisation ablation A–D)
**Does fine-tuning actually improve Recall@K?** To compare fairly and cheaply, all methods use the **same
mini-benchmark**: `mini_n_queries` validation S1 records, and per source an index of their true matches plus
`mini_n_distractors` random records. Retrieval is the same per-country top-K search. TF-IDF uses an exact sparse
dot product, which equals cosine because rows are L2-normalised.

The mini index is much smaller than the real one, so **absolute** recall is optimistic. Compare the **relative**
ordering of methods. The full-index numbers for the fine-tuned model are in Section 19. Set
`run_pretrained_full_baseline=True` to also get the pretrained model on the full index.

Ablation views: **A** name only · **B** address only · **C** name + address · **D** name + address + country
(primary). The ablation uses the pretrained encoder, so it needs no extra training runs.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

_rng = np.random.default_rng(SEED + 7)
MINI_Q = np.sort(_rng.choice(VAL_S1_ROWS, min(CONFIG["mini_n_queries"], len(VAL_S1_ROWS)), replace=False))
EV_MINI = make_eval_set(MINI_Q)

def _mini_index(true_rows, n_total, n_distr):
    return np.union1d(true_rows, _rng.choice(n_total, min(n_distr, n_total), replace=False))

MINI_X2 = _mini_index(EV_MINI["o2"], n2, CONFIG["mini_n_distractors"])
MINI_X3 = _mini_index(EV_MINI["o3"], n3, CONFIG["mini_n_distractors"])
MINI_N_INDEX = (len(MINI_X2), len(MINI_X3))
MINI_K = max(CONFIG["mini_k_list"])
print(f"Mini-benchmark: {len(MINI_Q):,} queries | S2 index {len(MINI_X2):,} | S3 index {len(MINI_X3):,}")

def _remap(I, rows):
    return np.where(I >= 0, rows[np.where(I >= 0, I, 0)], -1).astype(np.int32)

def mini_embed(model, view):
    qt = serialize(s1.iloc[MINI_Q], view)
    x2t, x3t = serialize(s2.iloc[MINI_X2], view), serialize(s3.iloc[MINI_X3], view)
    qe = encode_texts(model, qt, verbose=False)
    I2, D2 = search_by_country(qe, S1_CC[MINI_Q], encode_texts(model, x2t, verbose=False), S2_CC[MINI_X2], MINI_K, "S2")
    I3, D3 = search_by_country(qe, S1_CC[MINI_Q], encode_texts(model, x3t, verbose=False), S3_CC[MINI_X3], MINI_K, "S3")
    return finish_retrieval(_remap(I2, MINI_X2), D2, _remap(I3, MINI_X3), D3, EV_MINI)

def sparse_topk_by_country(Q, q_cc, X, x_cc, k, chunk=256):
    I = np.full((Q.shape[0], k), -1, dtype=np.int32)
    D = np.full((Q.shape[0], k), -np.inf, dtype=np.float32)
    groups = [(c, np.flatnonzero(q_cc == c)) for c in np.unique(q_cc)] if PARTITION else [(None, np.arange(Q.shape[0]))]
    for c, qidx in groups:
        xidx = np.flatnonzero(x_cc == c) if c is not None else np.arange(X.shape[0])
        if len(xidx) == 0:
            xidx = np.arange(X.shape[0])
        Xt = X[xidx].T.tocsr()
        kc = min(k, len(xidx))
        for s in range(0, len(qidx), chunk):
            rows = qidx[s:s + chunk]
            S = (Q[rows] @ Xt).toarray()
            part = np.argpartition(-S, kc - 1, axis=1)[:, :kc]
            vals = np.take_along_axis(S, part, 1)
            o = np.argsort(-vals, axis=1)
            I[rows, :kc] = xidx[np.take_along_axis(part, o, 1)]
            D[rows, :kc] = np.take_along_axis(vals, o, 1)
    return I, D

def mini_tfidf(view=PRIMARY_VIEW):
    qt = serialize(s1.iloc[MINI_Q], view)
    x2t, x3t = serialize(s2.iloc[MINI_X2], view), serialize(s3.iloc[MINI_X3], view)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=CONFIG["tfidf_ngram_range"], min_df=2,
                          sublinear_tf=True, dtype=np.float32)
    vec.fit(np.concatenate([x2t, x3t]))
    Q = vec.transform(qt)
    I2, D2 = sparse_topk_by_country(Q, S1_CC[MINI_Q], vec.transform(x2t), S2_CC[MINI_X2], MINI_K)
    I3, D3 = sparse_topk_by_country(Q, S1_CC[MINI_Q], vec.transform(x3t), S3_CC[MINI_X3], MINI_K)
    return finish_retrieval(_remap(I2, MINI_X2), D2, _remap(I3, MINI_X3), D3, EV_MINI)

MINI_RESULTS = {}
if CONFIG["run_tfidf_baseline"]:
    t = time.time(); MINI_RESULTS["TF-IDF char(2-4) | D"] = mini_tfidf(); print(f"TF-IDF done in {time.time()-t:.0f}s")

pretrained = make_encoder(MODEL_NAME)
for view in (VIEWS if CONFIG["run_ablation"] else [PRIMARY_VIEW]):
    MINI_RESULTS[f"Pretrained | {view}"] = mini_embed(pretrained, view)
MINI_RESULTS[f"Fine-tuned | {PRIMARY_VIEW}"] = mini_embed(encoder, PRIMARY_VIEW)

rows = []
for name, R in MINI_RESULTS.items():
    for k in CONFIG["mini_k_list"]:
        rows.append({"method": name, "K": k, **evaluate(EV_MINI, R, k, k, n_index=MINI_N_INDEX)})
BASELINE_DF = pd.DataFrame(rows)
BASELINE_DF.to_csv(OUT_DIR / "baseline_comparison_mini.csv", index=False)

pivot = BASELINE_DF.pivot(index="method", columns="K", values="pair_recall")
pivot.columns = [f"Recall@{k}" for k in pivot.columns]
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    print(pivot.sort_values(pivot.columns[2]))
    print("\nPer-source recall @10:")
    print(BASELINE_DF[BASELINE_DF.K == 10].set_index("method")[["s2_recall", "s3_recall", "mean_s1_recall",
                                                              "full_recall_pct"]])

PRETRAINED_FULL = None
if CONFIG["run_pretrained_full_baseline"]:
    Rp = run_validation_retrieval(pretrained, "pretrained", keep_emb=False)
    PRETRAINED_FULL = pd.DataFrame([
        {"model": "Pretrained (full index)", **evaluate(EV, Rp, CAND_CFG["k_s2"], CAND_CFG["k_s3"])},
        {"model": "Fine-tuned (full index)", **evaluate(EV, R_VAL, CAND_CFG["k_s2"], CAND_CFG["k_s3"])}])
    print("\nFull-index comparison at the chosen flat per-source K:")
    show(PRETRAINED_FULL, ["model", "k_s2", "k_s3", "pair_recall", "s2_recall", "s3_recall", "avg_cands"])
    del Rp
del pretrained
free_memory()

## 21. Retrieval failure analysis
For true pairs missed by the chosen configuration, we (a) tag likely causes and count them (tags can overlap), and
(b) print concrete examples. Each example shows the S1 record, **every** true match with its rank and cosine
(`rank=None` means it is outside K_max), and the top retrieved candidates with their similarity. These failures
tell us what to fix next.

In [ ]:
DEVA = r"[ऀ-ॿ]"

def _cfg_args(cfg):
    thr = cfg["sim_floor"] if cfg["mode"] == "dynamic" else None
    k2 = cfg["dyn_k"] if cfg["mode"] == "dynamic" else cfg["k_s2"]
    k3 = cfg["dyn_k"] if cfg["mode"] == "dynamic" else cfg["k_s3"]
    return k2, k3, thr, cfg["k_min"]

def _src_df(src):
    return s2 if src == 2 else s3

def missed_pairs(R, ev, cfg):
    k2, k3, thr, kmin = _cfg_args(cfg)
    out = []
    for src, k in ((2, k2), (3, k3)):
        m = ~_kept(R[f"r{src}"], R[f"s{src}"], k, thr, kmin)
        out.append(pd.DataFrame({"q": ev[f"q{src}"][m], "src": src, "o": ev[f"o{src}"][m],
                                 "rank": R[f"r{src}"][m]}))
    return pd.concat(out, ignore_index=True)

def tag_failures(miss, ev, max_n=100_000):
    if len(miss) == 0:
        print("No missed pairs."); return
    miss = miss.sample(min(max_n, len(miss)), random_state=0)
    s1r = ev["s1_rows"][miss["q"].values]
    a_name = s1["business_name"].values[s1r]; a_addr = s1["business_address"].values[s1r]
    b_name = np.where(miss.src == 2, s2["business_name"].values[np.where(miss.src == 2, miss.o, 0)],
                      s3["business_name"].values[np.where(miss.src == 3, miss.o, 0)])
    b_addr = np.where(miss.src == 2, s2["business_address"].values[np.where(miss.src == 2, miss.o, 0)],
                      s3["business_address"].values[np.where(miss.src == 3, miss.o, 0)])
    na, nb = normalize_series(a_name), normalize_series(b_name)
    jac = np.array([len(set(x.split()) & set(y.split())) / max(len(set(x.split()) | set(y.split())), 1)
                    for x, y in zip(na, nb)])
    tags = pd.DataFrame({
        "missing_address(either)": (pd.Series(a_addr) == "").values | (pd.Series(b_addr) == "").values,
        "script_mismatch(devanagari)": pd.Series(a_name).str.contains(DEVA).values != pd.Series(b_name).str.contains(DEVA).values,
        "same_norm_name": (na == nb).values,
        "low_name_token_jaccard(<0.3)": jac < 0.3,
        "outside_K_max": (miss["rank"].values == BIG_RANK),
    })
    print(f"Missed pairs: {len(miss):,} analysed")
    print((tags.mean() * 100).round(2).astype(str).add(" %").to_string())

def show_failures(R, ev, cfg, n_show):
    miss = missed_pairs(R, ev, cfg)
    tag_failures(miss, ev)
    if len(miss) == 0:
        return
    qs = np.random.default_rng(1).choice(miss["q"].unique(), min(n_show, miss["q"].nunique()), replace=False)
    for qi in qs:
        r1 = ev["s1_rows"][qi]
        print("=" * 110)
        print(f"S1 {s1.entity_id[r1]} | name={s1.business_name[r1]!r} | address={s1.business_address[r1]!r} "
              f"| country={s1.country[r1]}")
        print("  TRUE MATCHES:")
        for src in (2, 3):
            df = _src_df(src)
            for j in np.flatnonzero(ev[f"q{src}"] == qi):
                o, rk = ev[f"o{src}"][j], R[f"r{src}"][j]
                emb = R.get(f"e{src}")
                sim = float(R["q_emb"][qi].astype(np.float32) @ emb[o].astype(np.float32)) if emb is not None else np.nan
                status = "MISSED" if ((miss.q == qi) & (miss.src == src) & (miss.o == o)).any() else "found "
                print(f"    [{status}] {df.entity_id[o]} rank={None if rk == BIG_RANK else int(rk)} cos={sim:.3f} "
                      f"| {df.business_name[o]!r} | {df.business_address[o]!r} | {df.country[o]}")
        print("  TOP RETRIEVED:")
        for src in (2, 3):
            df = _src_df(src)
            for o, d in zip(R[f"I{src}"][qi, :4], R[f"D{src}"][qi, :4]):
                if o >= 0:
                    print(f"    S{src} cos={d:.3f} {df.entity_id[o]} | {df.business_name[o]!r} | "
                          f"{df.business_address[o]!r} | {df.country[o]}")

show_failures(R_VAL, EV, CAND_CFG, CONFIG["n_failure_examples"])

## 22. OPTIONAL — hard-negative mining (phase 3)
Off by default (`run_hard_negatives=False`). Turn it on once the baseline has been validated and the failure
analysis shows confusions such as *same name, different address*.

1. Sample `hn_n_anchors` train entity groups. Retrieve the top `hn_search_k` S2 and S3 neighbours of their S1 record
   with the **current** encoder.
2. Drop known positives (same component) and anything belonging to a **validation** component, which keeps
   validation clean.
3. Keep the `hn_per_anchor` most similar remaining records as hard negatives. They are appended to the group's
   batch, so the anchor sees e.g. *ABC Restaurant, MG Road* vs *ABC Restaurant, Brigade Road*.
4. Continue training, re-run validation retrieval, and **adopt the new model only if** it reaches the target with
   fewer candidates (or, if the target is unreachable, with higher recall at K_max). Otherwise the weights are
   restored.

S1 is deduplicated and its GT lists are complete, so a highly similar S2 record that is not in the anchor's list
is a real non-match *for that anchor*. It may still match a different S1, and it keeps that S1's label, so it
could appear as a positive of that S1 if both are in the same batch.

In [ ]:
def config_cost(sel):
    ch = sel["chosen"]
    return (0, ch["avg_cands"]) if sel["cfg"]["target_reached"] else (1, -ch["pair_recall"])

HARD_NEGS = None
if CONFIG["run_hard_negatives"]:
    pre_state = {k: v.detach().cpu().clone() for k, v in unwrap(encoder).backbone.state_dict().items()}
    rng = np.random.default_rng(SEED + 11)
    all_train_comps = np.unique(np.concatenate([t[1] for t in TRAIN_TABLES]))
    anchor_comps = rng.choice(all_train_comps, min(CONFIG["hn_n_anchors"], len(all_train_comps)), replace=False)
    first_s1 = np.full(N_COMP, -1, dtype=np.int64)
    first_s1[S1_COMP[::-1]] = np.arange(n1)[::-1]
    anchor_rows = first_s1[anchor_comps]

    qa = encode_texts(encoder, TEXT[anchor_rows], desc="HN anchors")
    e2 = R_VAL["e2"] if R_VAL.get("e2") is not None else encode_texts(encoder, s2["text"].values, desc="HN S2")
    I2h, D2h = search_by_country(qa, S1_CC[anchor_rows], e2, S2_CC, CONFIG["hn_search_k"], "S2")
    del e2; R_VAL["e2"] = None; free_memory()
    e3 = R_VAL["e3"] if R_VAL.get("e3") is not None else encode_texts(encoder, s3["text"].values, desc="HN S3")
    I3h, D3h = search_by_country(qa, S1_CC[anchor_rows], e3, S3_CC, CONFIG["hn_search_k"], "S3")
    del e3; R_VAL["e3"] = None; free_memory()

    nodes = np.concatenate([np.where(I2h >= 0, OFF2 + I2h.astype(np.int64), -1),
                            np.where(I3h >= 0, OFF3 + I3h.astype(np.int64), -1)], axis=1)
    sims = np.concatenate([D2h, D3h], axis=1)
    safe = np.where(nodes >= 0, nodes, 0)
    bad = (nodes < 0) | (NODE_COMP[safe] == anchor_comps[:, None]) | (COMP_SPLIT[NODE_COMP[safe]] == 1)
    sims[bad] = -np.inf
    top = np.argsort(-sims, axis=1)[:, :CONFIG["hn_per_anchor"]]
    HARD_NEGS = {}
    for i, c in enumerate(anchor_comps):
        sel = top[i][np.isfinite(sims[i, top[i]])]
        if len(sel):
            HARD_NEGS[int(c)] = nodes[i, sel]
    print(f"Hard negatives mined for {len(HARD_NEGS):,} groups "
          f"(mean cosine {np.nanmean(np.where(np.isfinite(sims[:, 0]), sims[:, 0], np.nan)):.3f})")
    for c in list(HARD_NEGS)[:3]:
        m = ORDER[COMP_OFFSETS[c]:COMP_OFFSETS[c + 1]]
        print("  anchor:", TEXT[m[m < n1][0]])
        for hn in HARD_NEGS[c]:
            print("     vs :", TEXT[hn])

    train_encoder(encoder, TRAIN_TABLES, epochs=CONFIG["hn_epochs"], max_minutes=CONFIG["hn_max_train_minutes"],
                  lr=CONFIG["lr"] * 0.5, run_name="hard_negative_run", neg_pool=TRAIN_S1_ROWS, hard_negs=HARD_NEGS)
    R_HN = run_validation_retrieval(encoder, "fine-tuned+HN", keep_emb=CONFIG["keep_val_embeddings"])
    SEL_HN = select_candidate_config(R_HN)
    print(f"\nBaseline cost {config_cost(SEL)} vs hard-negative cost {config_cost(SEL_HN)} (lower is better)")
    if config_cost(SEL_HN) < config_cost(SEL):
        print("-> ADOPTING hard-negative model.")
        R_VAL, SEL, CAND_CFG = R_HN, SEL_HN, SEL_HN["cfg"]
        SEL["sweep"].to_csv(OUT_DIR / "recall_at_k.csv", index=False)
        SEL["curves"].to_csv(OUT_DIR / "recall_curve_per_source.csv", index=False)
        SEL["thr"].to_csv(OUT_DIR / "similarity_threshold_sweep.csv", index=False)
    else:
        print("-> Keeping baseline model (restoring weights).")
        unwrap(encoder).backbone.load_state_dict(pre_state)
        HARD_NEGS = None
        del R_HN
    del pre_state
    free_memory()
else:
    print("Hard-negative phase skipped (run_hard_negatives=False).")

## 23. Exact normalised-key union + final validation numbers
Two cheap hash-join candidate sources:
* **exact_full**: identical normalised `(name, address, country)`, ignoring keys shared by > `exact_full_key_cap`
  index records.
* **exact_name**: identical normalised `(name, country)` for rare names only (≤ `exact_name_key_cap` records). This
  catches matches whose addresses differ or are missing.

Each is added to the final candidate set **only if** it raises validated pair recall by at least
`union_min_recall_gain`. The extra candidate cost is reported next to it. The resulting configuration is saved to
`candidate_config.json` and applied unchanged to the test set.

In [ ]:
def exact_pairs(q_keys, x_keys, cap):
    q = pd.DataFrame({"key": q_keys, "q": np.arange(len(q_keys), dtype=np.int64)})
    q = q[q["key"] != 0]
    x = pd.DataFrame({"key": x_keys, "o": np.arange(len(x_keys), dtype=np.int64)})
    x = x[x["key"] != 0]
    vc = x["key"].value_counts()
    x = x[x["key"].map(vc).values <= cap]
    m = q.merge(x, on="key", how="inner")
    return m["q"].to_numpy(), m["o"].to_numpy()

def build_extra(R, parts, n_index):
    """parts: list of {src: (q, o)}; returns {src: (q, o, rank_in_R, sim_in_R)} deduplicated."""
    extra = {}
    for src in (2, 3):
        qs = [p[src][0] for p in parts]; os_ = [p[src][1] for p in parts]
        if not qs:
            continue
        q, o = np.concatenate(qs), np.concatenate(os_)
        nx = n_index[src - 2]
        _, idx = np.unique(q.astype(np.int64) * nx + o, return_index=True)
        q, o = q[idx], o[idx]
        r, s = ranks_of(R[f"I{src}"], R[f"D{src}"], q, o) if R is not None else (None, None)
        extra[src] = (q, o, r, s)
    return extra

VAL_EXACT = {}
for key, cap in (("key_full", CONFIG["exact_full_key_cap"]), ("key_name", CONFIG["exact_name_key_cap"])):
    qk = s1[key].to_numpy()[VAL_S1_ROWS]
    VAL_EXACT[key] = {2: exact_pairs(qk, s2[key].to_numpy(), cap), 3: exact_pairs(qk, s3[key].to_numpy(), cap)}
    print(f"{key}: {len(VAL_EXACT[key][2][0]) + len(VAL_EXACT[key][3][0]):,} exact candidate pairs on validation")

k2, k3, thr, kmin = _cfg_args(CAND_CFG)
rows, selected = [], []
base = evaluate(EV, R_VAL, k2, k3, thr, kmin)
rows.append({"candidate_set": "FAISS only", **base})
best_recall = base["pair_recall"]
for key in ("key_full", "key_name"):
    trial = selected + [key]
    ex = build_extra(R_VAL, [VAL_EXACT[t] for t in trial], (n2, n3))
    m = evaluate(EV, R_VAL, k2, k3, thr, kmin, extra=ex)
    rows.append({"candidate_set": "FAISS + " + " + ".join(trial), **m})
    if m["pair_recall"] - best_recall >= CONFIG["union_min_recall_gain"]:
        selected, best_recall = trial, m["pair_recall"]
CAND_CFG["exact_union"] = selected
UNION_DF = pd.DataFrame(rows)
show(UNION_DF, ["candidate_set", "pair_recall", "s2_recall", "s3_recall", "full_recall_pct", "avg_cands",
                "avg_cands_singleton", "pair_precision"])
print(f"\nExact-key sources selected for the final candidate set: {selected or 'none'}")

# Final validation metrics with the exact configuration that will be applied to test.
VAL_EXTRA = build_extra(R_VAL, [VAL_EXACT[t] for t in selected], (n2, n3)) if selected else None
FINAL_VAL_METRICS, PERQ = evaluate(EV, R_VAL, k2, k3, thr, kmin, extra=VAL_EXTRA, per_query=True)
print("\nFINAL validation candidate metrics:")
print(json.dumps({k: (round(v, 5) if isinstance(v, float) else v) for k, v in FINAL_VAL_METRICS.items()}, indent=1))

# ---- validation_retrieval_details.csv (one row per validation S1) ----
nt2 = np.bincount(EV["q2"], minlength=EV["nq"]); nt3 = np.bincount(EV["q3"], minlength=EV["nq"])
val_details = pd.DataFrame({
    "source1_entity_id": s1["entity_id"].values[VAL_S1_ROWS],
    "country": s1["country"].values[VAL_S1_ROWS],
    "is_singleton": EV["n_true"] == 0,
    "n_true": EV["n_true"], "n_true_s2": nt2, "n_true_s3": nt3,
    "n_true_retrieved": PERQ["n_hit"].astype(int),
    "recall": np.where(EV["n_true"] > 0, PERQ["n_hit"] / np.maximum(EV["n_true"], 1), np.nan),
    "n_candidates": PERQ["n_cand"], "n_candidates_s2": PERQ["n_cand_s2"], "n_candidates_s3": PERQ["n_cand_s3"],
    "top1_cos_s2": R_VAL["D2"][:, 0], "top1_cos_s3": R_VAL["D3"][:, 0],
})
val_details.to_csv(OUT_DIR / "validation_retrieval_details.csv", index=False)
json.dump({**CAND_CFG, "model_name": MODEL_NAME, "partition_by_country": PARTITION,
           "validation_metrics": {k: (float(v) if v is not None else None) for k, v in FINAL_VAL_METRICS.items()}},
          open(OUT_DIR / "candidate_config.json", "w"), indent=1, default=str)
print(f"\nSaved {OUT_DIR/'validation_retrieval_details.csv'} and {OUT_DIR/'candidate_config.json'}")
print(f"Projected test candidate pairs: ~{FINAL_VAL_METRICS['avg_cands'] * 1_732_544:,.0f} "
      f"(avg {FINAL_VAL_METRICS['avg_cands']:.1f} x 1.73M test S1)")

## 23b. Candidate pruner (learned filter → final, smaller candidate set)
The organisers rank solutions with **smaller candidate sets per S1** higher, and `candidate_pairs.tsv` must be the
*last* filter before the matcher. So after FAISS retrieval (the recall-oriented **pool**, tuned to `target_recall`),
a **lightweight learned pruner** removes candidates that are clearly not matches:

* **Features** (cheap, per pair): cosine, rank, cosine − top-1 cosine (same source / both sources), the query's top-1
  cosine and top-1/top-2 gap, source flag, rapidfuzz name/address `ratio` and `token_set_ratio`, house-number / PIN
  / ZIP token agreement, missing-address flags, Devanagari script mismatch, name length difference.
* **Model:** LightGBM (fallback: sklearn HistGradientBoosting). No external data.
* **Honest tuning:** validation queries are split in two halves: the pruner is **fit on half A**, and the score
  threshold is chosen on **half B** as the one that gives the **fewest candidates** while keeping final pair recall
  ≥ `prune_target_recall`, measured against *all* true pairs (including those FAISS already missed).
* **Singletons** can now get an **empty** candidate list when nothing scores high enough.
* **Countries unseen in training (France)** use a safer threshold tuned for `prune_unseen_target_recall`.
* Exact-key candidates (if selected in Section 23) bypass the pruner and are always kept.

The pruner is part of **blocking**. The next-stage matcher still makes the final match decision, on this smaller set.

In [ ]:
PRUNE_FEATURES = ["cos", "rank", "cos_minus_top1_src", "cos_minus_top1_all", "top1_all", "q_gap12_src", "is_s3",
                  "name_ratio", "name_token_set", "addr_ratio", "addr_token_set", "digits_token_set",
                  "q_addr_missing", "x_addr_missing", "script_mismatch", "name_len_diff"]

def _finite_or_nan(a):
    a = np.asarray(a, dtype=np.float32).copy()
    a[~np.isfinite(a)] = np.nan
    return a

def query_top_stats(D):
    with np.errstate(invalid="ignore"):
        top1 = _finite_or_nan(D[:, 0])
        gap = _finite_or_nan(D[:, 0] - D[:, 1]) if D.shape[1] > 1 else np.full(len(D), np.nan, np.float32)
    return top1, gap

def _rf(a, b, scorer):
    return rf_process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32) / 100.0

def pair_features(qdf, qrows, xdf, orows, src, cos, rank, top1_src, gap_src, top1_all):
    qn = qdf["norm_name"].to_numpy()[qrows]; xn = xdf["norm_name"].to_numpy()[orows]
    qa = qdf["norm_address"].to_numpy()[qrows]; xa = xdf["norm_address"].to_numpy()[orows]
    qd = qdf["digits"].to_numpy()[qrows]; xd = xdf["digits"].to_numpy()[orows]
    qa_miss, xa_miss = (qa == ""), (xa == "")
    d_miss = (qd == "") | (xd == "")
    cos = _finite_or_nan(cos)
    F = np.empty((len(qrows), len(PRUNE_FEATURES)), dtype=np.float32)
    F[:, 0] = cos
    F[:, 1] = rank
    F[:, 2] = cos - top1_src
    F[:, 3] = cos - top1_all
    F[:, 4] = top1_all
    F[:, 5] = gap_src
    F[:, 6] = float(src == 3)
    qn_l, xn_l, qa_l, xa_l = qn.tolist(), xn.tolist(), qa.tolist(), xa.tolist()
    F[:, 7] = _rf(qn_l, xn_l, fuzz.ratio)
    F[:, 8] = _rf(qn_l, xn_l, fuzz.token_set_ratio)
    ar, ats = _rf(qa_l, xa_l, fuzz.ratio), _rf(qa_l, xa_l, fuzz.token_set_ratio)
    ar[qa_miss | xa_miss] = np.nan; ats[qa_miss | xa_miss] = np.nan
    F[:, 9], F[:, 10] = ar, ats
    dts = _rf(qd.tolist(), xd.tolist(), fuzz.token_set_ratio)
    dts[d_miss] = np.nan
    F[:, 11] = dts
    F[:, 12] = qa_miss
    F[:, 13] = xa_miss
    F[:, 14] = qdf["is_deva"].to_numpy()[qrows] != xdf["is_deva"].to_numpy()[orows]
    F[:, 15] = np.abs(np.fromiter(map(len, qn_l), np.int32, len(qn_l)) - np.fromiter(map(len, xn_l), np.int32, len(xn_l)))
    return F

def score_pairs(model, qdf, qrows, xdf, orows, src, cos, rank, top1_src, gap_src, top1_all, chunk=2_000_000,
                desc="pruner"):
    out = np.empty(len(orows), dtype=np.float32)
    for a in tqdm(range(0, len(orows), chunk), desc=desc, disable=len(orows) <= chunk):
        b = a + chunk
        X = pair_features(qdf, qrows[a:b], xdf, orows[a:b], src, cos[a:b], rank[a:b],
                          top1_src[a:b], gap_src[a:b], top1_all[a:b])
        out[a:b] = model.predict_proba(X)[:, 1]
    return out

def make_pruner_model():
    try:
        import lightgbm as lgb
        return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                                  subsample=0.8, subsample_freq=1, colsample_bytree=0.9, n_jobs=-1,
                                  random_state=SEED, verbose=-1), "LightGBM"
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=63,
                                              random_state=SEED), "HistGradientBoosting"

def stage1_pairs_val(R, qpos, cfg):
    k2, k3, thr, kmin = _cfg_args(cfg)
    out = []
    for src, k in ((2, k2), (3, k3)):
        I, D = R[f"I{src}"][qpos], R[f"D{src}"][qpos]
        k = min(k, I.shape[1]); I, D = I[:, :k], D[:, :k]
        m = I >= 0
        if thr is not None:
            m &= (D >= thr) | (np.arange(k) < kmin)[None, :]
        qi, col = np.nonzero(m)
        out.append({"src": src, "q": qpos[qi], "o": I[qi, col].astype(np.int64), "cos": D[qi, col],
                    "rank": col.astype(np.float32)})
    return out

PRUNER = PRUNE_TAU = PRUNE_TAU_UNSEEN = PRUNE_METRICS = PRUNE_STAGE1_METRICS = PRUNE_TRADEOFF = None
if CONFIG["use_pruner"]:
    t0 = time.time()
    if CONFIG["final_train_mode"] != "reuse":
        print("NOTE: final_train_mode != 'reuse' -> test cosines come from a slightly different model than the one "
              "the pruner was tuned on. 'reuse' keeps the pruner exactly calibrated.")
    NX = {2: n2, 3: n3}
    rng = np.random.default_rng(SEED + 21)
    qsel = rng.permutation(EV["nq"])[:min(CONFIG["prune_n_queries"], EV["nq"])]
    qA, qB = np.sort(qsel[:len(qsel) // 2]), np.sort(qsel[len(qsel) // 2:])
    V_STATS = {s: query_top_stats(R_VAL[f"D{s}"]) for s in (2, 3)}
    V_TOP1_ALL = np.fmax(V_STATS[2][0], V_STATS[3][0])
    TRUE_CODES = {s: EV[f"q{s}"].astype(np.int64) * NX[s] + EV[f"o{s}"] for s in (2, 3)}
    sel_keys = CAND_CFG.get("exact_union", [])
    EXACT_CODES = {s: (np.unique(np.concatenate([VAL_EXACT[t][s][0].astype(np.int64) * NX[s] + VAL_EXACT[t][s][1]
                                                 for t in sel_keys])) if sel_keys else np.zeros(0, np.int64))
                   for s in (2, 3)}

    def val_pairs_with_features(qpos, tag):
        parts = stage1_pairs_val(R_VAL, qpos, CAND_CFG)
        for p in parts:
            s, q = p["src"], p["q"]
            code = q.astype(np.int64) * NX[s] + p["o"]
            p["code"], p["y"], p["in_exact"] = code, np.isin(code, TRUE_CODES[s]), np.isin(code, EXACT_CODES[s])
            p["X"] = pair_features(s1, VAL_S1_ROWS[q], s2 if s == 2 else s3, p["o"], s, p["cos"], p["rank"],
                                   V_STATS[s][0][q], V_STATS[s][1][q], V_TOP1_ALL[q])
        print(f"  {tag}: {len(qpos):,} queries -> {sum(len(p['q']) for p in parts):,} stage-1 pairs "
              f"({sum(p['y'].sum() for p in parts):,} true)")
        return parts

    PA = val_pairs_with_features(qA, "fit half A")
    PB = val_pairs_with_features(qB, "threshold half B")
    PRUNER, pruner_kind = make_pruner_model()
    XA = np.concatenate([p["X"] for p in PA]); yA = np.concatenate([p["y"] for p in PA])
    PRUNER.fit(XA, yA)
    del XA, yA, PA
    for p in PB:
        p["score"] = PRUNER.predict_proba(p["X"])[:, 1]
        del p["X"]
    print(f"Pruner ({pruner_kind}) trained in {time.time()-t0:.0f}s")
    if hasattr(PRUNER, "feature_importances_"):
        imp = pd.Series(PRUNER.feature_importances_, index=PRUNE_FEATURES).sort_values(ascending=False)
        print("Feature importance:", ", ".join(f"{k}={v:.0f}" for k, v in imp.head(8).items()))

    # ---- evaluation on half B ----
    b_mask = np.zeros(EV["nq"], dtype=bool); b_mask[qB] = True
    nB = len(qB)
    true_B = {s: TRUE_CODES[s][b_mask[EV[f"q{s}"]]] for s in (2, 3)}
    total_true_B = sum(len(v) for v in true_B.values())
    exact_only_B = {}
    for s in (2, 3):
        ec = EXACT_CODES[s][b_mask[EXACT_CODES[s] // NX[s]]] if len(EXACT_CODES[s]) else EXACT_CODES[s]
        pool = next(p["code"] for p in PB if p["src"] == s)
        exact_only_B[s] = ec[~np.isin(ec, pool)]
    base_hits = sum(int(np.isin(true_B[s], EXACT_CODES[s]).sum()) for s in (2, 3))

    def prune_eval(tau, unseen_tau=None):
        n_hit = np.zeros(EV["nq"]); n_cand = np.zeros(EV["nq"]); hits = {}
        for p in PB:
            s = p["src"]
            keep = p["in_exact"] | (p["score"] >= tau)
            kept_codes = np.concatenate([p["code"][keep], exact_only_B[s]])
            h = np.isin(true_B[s], kept_codes)
            hits[s] = h
            np.add.at(n_hit, true_B[s][h] // NX[s], 1)
            n_cand += np.bincount(kept_codes // NX[s], minlength=EV["nq"])
        nt = EV["n_true"][qB]; nh = n_hit[qB]; nc = n_cand[qB]; ns = nt > 0
        return {"threshold": float(tau),
                "pair_recall": (hits[2].sum() + hits[3].sum()) / max(total_true_B, 1),
                "s2_recall": float(hits[2].mean()) if len(hits[2]) else np.nan,
                "s3_recall": float(hits[3].mean()) if len(hits[3]) else np.nan,
                "full_recall_pct": float((nh[ns] == nt[ns]).mean() * 100) if ns.any() else np.nan,
                "avg_cands": float(nc.mean()), "p95_cands": float(np.percentile(nc, 95)),
                "max_cands": float(nc.max()),
                "avg_cands_singleton": float(nc[~ns].mean()) if (~ns).any() else np.nan,
                "singleton_empty_pct": float((nc[~ns] == 0).mean() * 100) if (~ns).any() else np.nan,
                "empty_pct": float((nc == 0).mean() * 100)}

    free_s = np.concatenate([p["score"][~p["in_exact"]] for p in PB])
    free_y = np.concatenate([p["y"][~p["in_exact"]] for p in PB])
    o = np.argsort(-free_s, kind="stable")
    s_sorted, recall_sorted = free_s[o], (base_hits + np.cumsum(free_y[o])) / max(total_true_B, 1)
    stage1_recall_B = float(recall_sorted[-1]) if len(recall_sorted) else base_hits / max(total_true_B, 1)

    def tau_for(target):
        if target > stage1_recall_B:
            target = stage1_recall_B * CONFIG["prune_fallback_retention"]
        idx = int(np.argmax(recall_sorted >= target))
        return float(s_sorted[idx]), target

    PRUNE_STAGE1_METRICS = prune_eval(-np.inf)
    rows = [{"setting": "stage-1 pool (no pruning)", **PRUNE_STAGE1_METRICS}]
    for tgt in [0.95, 0.97, 0.98, 0.985, 0.99, 0.993, 0.995, 0.998]:
        if tgt < stage1_recall_B:
            tau, _ = tau_for(tgt)
            rows.append({"setting": f"recall>={tgt}", **prune_eval(tau)})
    PRUNE_TRADEOFF = pd.DataFrame(rows)
    PRUNE_TAU, eff_target = tau_for(CONFIG["prune_target_recall"])
    PRUNE_TAU_UNSEEN, eff_unseen = tau_for(CONFIG["prune_unseen_target_recall"])
    PRUNE_METRICS = prune_eval(PRUNE_TAU)
    PRUNE_TRADEOFF.to_csv(OUT_DIR / "pruner_tradeoff.csv", index=False)
    import pickle
    with open(OUT_DIR / "pruner.pkl", "wb") as f:
        pickle.dump({"model": PRUNER, "features": PRUNE_FEATURES, "threshold": PRUNE_TAU,
                     "threshold_unseen_country": PRUNE_TAU_UNSEEN}, f)

    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 250):
        print("\nPruning trade-off on held-out half B (recall is vs ALL true pairs):")
        print(PRUNE_TRADEOFF[["setting", "threshold", "pair_recall", "s2_recall", "s3_recall", "full_recall_pct",
                              "avg_cands", "p95_cands", "avg_cands_singleton", "singleton_empty_pct"]].to_string(index=False))
    print(f"\nSelected threshold {PRUNE_TAU:.4f} (target recall {eff_target:.4f}); unseen-country threshold "
          f"{PRUNE_TAU_UNSEEN:.4f} (target {eff_unseen:.4f})")
    print(f"Candidates per S1: {PRUNE_STAGE1_METRICS['avg_cands']:.1f} -> {PRUNE_METRICS['avg_cands']:.2f} | "
          f"pair recall {PRUNE_STAGE1_METRICS['pair_recall']:.4f} -> {PRUNE_METRICS['pair_recall']:.4f} | "
          f"singletons with empty list: {PRUNE_METRICS['singleton_empty_pct']:.1f}%")
    del PB, free_s, free_y, o
    free_memory()
    print(f"Pruner stage done in {(time.time()-t0)/60:.1f} min")
else:
    print("Pruner disabled (use_pruner=False): candidate_pairs.tsv = stage-1 pool.")

json.dump({**CAND_CFG, "model_name": MODEL_NAME, "partition_by_country": PARTITION,
           "stage1_validation_metrics": {k: (float(v) if v is not None else None) for k, v in FINAL_VAL_METRICS.items()},
           "pruner": None if PRUNER is None else {
               "threshold": PRUNE_TAU, "threshold_unseen_country": PRUNE_TAU_UNSEEN,
               "target_recall": CONFIG["prune_target_recall"], "features": PRUNE_FEATURES,
               "validation_metrics_half_B": PRUNE_METRICS, "stage1_metrics_half_B": PRUNE_STAGE1_METRICS}},
          open(OUT_DIR / "candidate_config.json", "w"), indent=1, default=str)

## 23c. Export labelled validation candidates (training data for the next-stage matcher)
The matcher must learn from **exactly the kind of candidates it will see on test**. We therefore export the final
(post-pruning) candidate set for every validation S1 that was **not** used to fit the pruner (its scores would be
optimistic), with labels, raw strings and all blocking scores:

* `val_candidate_pairs.parquet`: one row per (validation S1, candidate) with `label`, `cosine`, `faiss_rank`,
  `prune_score`, `via_exact`, `source`, raw name/address/country of both records.
* `val_queries.parquet`: one row per exported validation S1 with `n_true` (**all** true matches, including those
  blocking missed). This is needed to compute the exact per-S1 F0.5, singletons included.

In [ ]:
if CONFIG.get("export_val_candidates", True):
    t0 = time.time()
    NX = {2: n2, 3: n3}
    TRUE_CODES = {s: EV[f"q{s}"].astype(np.int64) * NX[s] + EV[f"o{s}"] for s in (2, 3)}
    sel_keys = CAND_CFG.get("exact_union", [])
    EXACT_CODES = {s: (np.unique(np.concatenate([VAL_EXACT[t][s][0].astype(np.int64) * NX[s] + VAL_EXACT[t][s][1]
                                                 for t in sel_keys])) if sel_keys else np.zeros(0, np.int64))
                   for s in (2, 3)}
    if PRUNER is not None:
        V_STATS = {s: query_top_stats(R_VAL[f"D{s}"]) for s in (2, 3)}
        V_TOP1_ALL = np.fmax(V_STATS[2][0], V_STATS[3][0])
    exp_q = np.setdiff1d(np.arange(EV["nq"]), qA) if PRUNER is not None else np.arange(EV["nq"])
    exp_mask = np.zeros(EV["nq"], dtype=bool); exp_mask[exp_q] = True
    k2, k3, thr, kmin = _cfg_args(CAND_CFG)
    cols = {k: [] for k in ("q", "src", "o", "cosine", "faiss_rank", "prune_score", "via_exact")}

    def _add(q, src, o, cos, rank, score, via):
        cols["q"].append(q.astype(np.int64)); cols["src"].append(np.full(len(q), src, np.int8))
        cols["o"].append(o.astype(np.int64)); cols["cosine"].append(np.asarray(cos, np.float32))
        cols["faiss_rank"].append(np.asarray(rank, np.int32)); cols["prune_score"].append(np.asarray(score, np.float32))
        cols["via_exact"].append(np.asarray(via, bool))

    for a in tqdm(range(0, len(exp_q), 50_000), desc="export val candidates"):
        qpos = exp_q[a:a + 50_000]
        for p in stage1_pairs_val(R_VAL, qpos, CAND_CFG):
            s, q = p["src"], p["q"]
            code = q.astype(np.int64) * NX[s] + p["o"]
            in_exact = np.isin(code, EXACT_CODES[s])
            if PRUNER is not None:
                score = score_pairs(PRUNER, s1, VAL_S1_ROWS[q], s2 if s == 2 else s3, p["o"], s, p["cos"], p["rank"],
                                    V_STATS[s][0][q], V_STATS[s][1][q], V_TOP1_ALL[q])
                keep = in_exact | (score >= PRUNE_TAU)
            else:
                score = np.full(len(q), np.nan, np.float32)
                keep = np.ones(len(q), dtype=bool)
            _add(q[keep], s, p["o"][keep], p["cos"][keep], p["rank"][keep], score[keep], in_exact[keep])
    # exact-key pairs outside the FAISS pool (always kept, like on test)
    for s, k in ((2, k2), (3, k3)):
        ec = EXACT_CODES[s]
        if len(ec):
            eq, eo = ec // NX[s], ec % NX[s]
            m = exp_mask[eq]; eq, eo = eq[m], eo[m]
            r, sm = ranks_of(R_VAL[f"I{s}"], R_VAL[f"D{s}"], eq, eo)
            only = ~_kept(r, sm, k, thr, kmin)
            _add(eq[only], s, eo[only], np.full(only.sum(), np.nan), np.full(only.sum(), -1),
                 np.full(only.sum(), np.nan), np.ones(only.sum(), bool))

    V = {k: np.concatenate(v) for k, v in cols.items()}
    s1r = VAL_S1_ROWS[V["q"]]
    code = V["q"] * np.where(V["src"] == 2, n2, n3) + V["o"]
    label = np.where(V["src"] == 2, np.isin(code, TRUE_CODES[2]), np.isin(code, TRUE_CODES[3]))
    o2 = np.where(V["src"] == 2, V["o"], 0); o3 = np.where(V["src"] == 3, V["o"], 0)
    pick = lambda col: np.where(V["src"] == 2, s2[col].to_numpy(object)[o2], s3[col].to_numpy(object)[o3])
    val_pairs = pd.DataFrame({
        "source1_entity_id": s1["entity_id"].to_numpy(object)[s1r],
        "candidate_entity_id": pick("entity_id"),
        "source": np.where(V["src"] == 2, "S2", "S3"),
        "cosine": V["cosine"], "faiss_rank": V["faiss_rank"], "prune_score": V["prune_score"],
        "via_exact": V["via_exact"], "label": label,
        "s1_name": s1["business_name"].to_numpy(object)[s1r], "s1_address": s1["business_address"].to_numpy(object)[s1r],
        "s1_country": s1["country"].to_numpy(object)[s1r],
        "cand_name": pick("business_name"), "cand_address": pick("business_address"), "cand_country": pick("country"),
    })
    nt2 = np.bincount(EV["q2"], minlength=EV["nq"]); nt3 = np.bincount(EV["q3"], minlength=EV["nq"])
    val_queries = pd.DataFrame({
        "source1_entity_id": s1["entity_id"].to_numpy(object)[VAL_S1_ROWS[exp_q]],
        "country": s1["country"].to_numpy(object)[VAL_S1_ROWS[exp_q]],
        "n_true": EV["n_true"][exp_q], "n_true_s2": nt2[exp_q], "n_true_s3": nt3[exp_q],
    })
    val_pairs.to_parquet(OUT_DIR / "val_candidate_pairs.parquet", index=False)
    val_queries.to_parquet(OUT_DIR / "val_queries.parquet", index=False)
    rec = label.sum() / max(val_queries["n_true"].sum(), 1)
    nc = np.bincount(V["q"], minlength=EV["nq"])[exp_q]
    sing = val_queries["n_true"].values == 0
    print(f"Exported {len(val_pairs):,} validation candidate pairs for {len(val_queries):,} S1 "
          f"({len(val_pairs)/max(len(val_queries),1):.2f}/S1, positives {label.mean():.2%}) in {(time.time()-t0)/60:.1f} min")
    print(f"Blocking recall on exported queries: {rec:.4f} | singletons with empty list: "
          f"{(nc[sing] == 0).mean():.1%} | recall ceiling for the matcher = {rec:.4f}")
    del V, val_pairs, s1r, code, label
    free_memory()

## 24. Final encoder
Validation has served its purpose, so the final encoder should also learn from the validation groups:
* `"continue"` (default): keep training the validated model on **train + val** groups for `final_epochs` at
  `lr × final_lr_scale`. This is cheap, and the K and floor chosen on validation stay approximately valid.
* `"retrain"`: train from the pretrained checkpoint on all groups with the original budget. This is the most
  principled option and the most expensive.
* `"reuse"`: use the validated model as-is. It is fastest, and K/floor are exactly calibrated for it.

Note: any extra training slightly shifts the similarity scale. If dynamic mode was selected and you want the
floor to be exactly calibrated, use `"reuse"`.

In [ ]:
ALL_TABLES = [("S1<->S2 (train+val)", eligible_components([0, 1], CNT_S2), 2),
              ("S1<->S3 (train+val)", eligible_components([0, 1], CNT_S3), 3)]
mode = CONFIG["final_train_mode"]
N_TRAIN_GROUPS = {"validation_run": int(sum(len(t[1]) for t in TRAIN_TABLES)),
                  "final": int(sum(len(t[1]) for t in ALL_TABLES)) if mode != "reuse" else 0}

# Free validation-phase memory before more training / test encoding.
for _k in ("e2", "e3", "q_emb"):
    R_VAL[_k] = None
MINI_RESULTS = None
free_memory()

if mode == "retrain":
    final_encoder = make_encoder(MODEL_NAME)
    train_encoder(final_encoder, ALL_TABLES, epochs=CONFIG["epochs"], max_minutes=CONFIG["max_train_minutes"],
                  lr=CONFIG["lr"], run_name="final_retrain", neg_pool=np.arange(n1), hard_negs=HARD_NEGS)
elif mode == "continue":
    final_encoder = encoder
    train_encoder(final_encoder, ALL_TABLES, epochs=CONFIG["final_epochs"],
                  max_minutes=CONFIG["final_max_train_minutes"], lr=CONFIG["lr"] * CONFIG["final_lr_scale"],
                  run_name="final_continue", neg_pool=np.arange(n1), hard_negs=HARD_NEGS)
elif mode == "reuse":
    final_encoder = encoder
else:
    raise ValueError(f"unknown final_train_mode {mode!r}")

unwrap(final_encoder).backbone.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
json.dump({"base_model": MODEL_NAME, "license": CONFIG["model_license"], "pooling": CONFIG["pooling"] + " + L2 normalise",
           "max_seq_len": CONFIG["max_seq_len"], "serialization_view": PRIMARY_VIEW,
           "final_train_mode": mode, "hard_negatives": HARD_NEGS is not None},
          open(MODEL_DIR / "encoder_info.json", "w"), indent=1)
print(f"Final encoder ({mode}) saved to {MODEL_DIR}")

# Training-side frames are no longer needed; keep only summary numbers.
TRAIN_SUMMARY = {"train_s1": n1, "train_s2": n2, "train_s3": n3, "val_s1": len(VAL_S1_ROWS),
                 "val_singletons": int((EV["n_true"] == 0).sum()),
                 "train_groups": N_TRAIN_GROUPS}
del s1, s2, s3, TEXT, ORDER, NODE_COMP, R_VAL, VAL_EXACT, VAL_EXTRA, PAIR_Q, PAIR_O, PAIR_NODE, PAIR_SRC
for _name in ("gt_raw", "pairs", "comp_df", "strat", "bucket"):
    globals().pop(_name, None)
free_memory()
print("RAM:", ram_gb())

## 25. Load test data
The test set adds **France**. Country is an open vocabulary: France gets its own code and its own FAISS partition,
and every test S1 row (France included) gets a row in the output. Countries unseen in training get flat
`K × unseen_country_k_multiplier` candidates and never a similarity floor, because we cannot validate them.

In [ ]:
print("Loading test data ...")
test_s1 = read_source(PATHS["test_s1"])
test_s2 = read_source(PATHS["test_s2"])
test_s3 = read_source(PATHS["test_s3"])
for name, df, prefix in [("test S1", test_s1, "S1-"), ("test S2", test_s2, "S2-"), ("test S3", test_s3, "S3-")]:
    assert df["entity_id"].str.startswith(prefix).all(), f"bad prefix in {name}"
    assert not df["entity_id"].duplicated().any(), f"duplicate ids in {name}"
N_TEST_S1_FULL = len(test_s1)

if DEBUG_MODE:
    rng = np.random.default_rng(SEED + 5)
    test_s1 = test_s1.iloc[np.sort(rng.choice(len(test_s1), min(CONFIG["debug_test_s1_n"], len(test_s1)), replace=False))]
    test_s2 = test_s2.iloc[np.sort(rng.choice(len(test_s2), min(CONFIG["debug_test_s2_n"], len(test_s2)), replace=False))]
    test_s3 = test_s3.iloc[np.sort(rng.choice(len(test_s3), min(CONFIG["debug_test_s3_n"], len(test_s3)), replace=False))]
    print(f"DEBUG: test subsample S1 {len(test_s1):,} / S2 {len(test_s2):,} / S3 {len(test_s3):,} "
          "(output will NOT be submittable)")

test_s1 = prepare_source(test_s1, "test S1")
test_s2 = prepare_source(test_s2, "test S2")
test_s3 = prepare_source(test_s3, "test S3")
tn1, tn2, tn3 = len(test_s1), len(test_s2), len(test_s3)
T1_CC, T2_CC, T3_CC = (d["country_code"].to_numpy() for d in (test_s1, test_s2, test_s3))

print("\nTest S1 country distribution:")
print(test_s1["country"].value_counts().to_string())
UNSEEN = sorted(set(np.unique(T1_CC).tolist()) - TRAIN_COUNTRY_CODES)
print(f"Countries unseen in training: {[country_name(c) for c in UNSEEN]}")
if "france" not in COUNTRY_VOCAB:
    print("NOTE: France not found in the loaded test S1 (expected per the spec; check the data if full mode).")

## 26–27. Encode test S1/S2/S3 and search the per-source FAISS indexes
The same functions as in validation (dedupe-before-encode, per-country exact search) with the candidate
configuration from Section 23. K is set **per country**: validated countries use `K_S2`/`K_S3` (or the similarity-floor
cap), and unseen countries use the boosted flat K. Exact-key pairs are computed while each source's embeddings are in
memory, so their cosine can be recorded for the matcher.

In [ ]:
def k_per_country(src):
    base = CAND_CFG["k_s2"] if src == 2 else CAND_CFG["k_s3"]
    cap = CONFIG.get("pool_k_cap") or CONFIG["k_max"]
    out = {}
    for c in np.unique(T1_CC):
        c = int(c)
        if c in TRAIN_COUNTRY_CODES:
            out[c] = CAND_CFG["dyn_k"] if CAND_CFG["mode"] == "dynamic" else base
        else:
            out[c] = int(min(max(cap, base), math.ceil(base * CONFIG["unseen_country_k_multiplier"])))
    return out

def keep_mask(I, D, k_dict, q_cc):
    klut = np.zeros(max(COUNTRY_VOCAB.values()) + 1, dtype=np.int32)
    for c, k in k_dict.items():
        klut[c] = k
    col = np.arange(I.shape[1])[None, :]
    m = (I >= 0) & (col < klut[q_cc][:, None])
    if CAND_CFG["mode"] == "dynamic":
        seen = np.isin(q_cc, list(TRAIN_COUNTRY_CODES))[:, None]
        m &= ~seen | (D >= CAND_CFG["sim_floor"]) | (col < CAND_CFG["k_min"])
    return m

def exact_pairs_test(src, q_emb, x_emb, exact_keys):
    """Exact-key candidates (always kept, bypass the pruner) with their cosine."""
    qs, os_, sims, vias = [], [], [], []
    for bit, key in ((2, "key_full"), (4, "key_name")):
        if key not in exact_keys:
            continue
        cap = CONFIG["exact_full_key_cap"] if key == "key_full" else CONFIG["exact_name_key_cap"]
        x_df = test_s2 if src == 2 else test_s3
        eq, eo = exact_pairs(test_s1[key].to_numpy(), x_df[key].to_numpy(), cap)
        es = np.concatenate([(q_emb[eq[a:a + 1_000_000]].astype(np.float32) *
                              x_emb[eo[a:a + 1_000_000]].astype(np.float32)).sum(1)
                             for a in range(0, len(eq), 1_000_000)]) if len(eq) else np.zeros(0, np.float32)
        qs.append(eq); os_.append(eo); sims.append(es.astype(np.float32)); vias.append(np.full(len(eq), bit, np.int8))
    if not qs:
        return None
    return {"q": np.concatenate(qs), "o": np.concatenate(os_), "sim": np.concatenate(sims),
            "rank": np.full(sum(len(x) for x in qs), BIG_RANK, np.int32), "via": np.concatenate(vias),
            "score": np.full(sum(len(x) for x in qs), np.nan, np.float32)}

def dedupe_pairs(P, nx):
    code_ = P["q"].astype(np.int64) * nx + P["o"]
    order = np.argsort(code_, kind="stable")
    code_s = code_[order]
    starts = np.flatnonzero(np.r_[True, code_s[1:] != code_s[:-1]])
    return {"q": P["q"][order][starts], "o": P["o"][order][starts],
            "sim": np.maximum.reduceat(P["sim"][order], starts),
            "rank": np.minimum.reduceat(P["rank"][order], starts),
            "via": np.bitwise_or.reduceat(P["via"][order], starts),
            "score": np.fmax.reduceat(P["score"][order], starts)}

# ---- 1) encode + search both sources; keep only the (n_queries x K) neighbour arrays ----
t = time.time()
te1 = encode_texts(final_encoder, test_s1["text"].values, desc="test S1")
SEARCH, T_STATS, EXACT_TEST = {}, {}, {}
for src, df, cc in ((2, test_s2, T2_CC), (3, test_s3, T3_CC)):
    kd = k_per_country(src)
    print(f"S{src} K per country: { {country_name(c): k for c, k in kd.items()} }")
    xe = encode_texts(final_encoder, df["text"].values, desc=f"test S{src}")
    I, D = search_by_country(te1, T1_CC, xe, cc, kd, desc=f"S{src}")
    SEARCH[src] = (I, D, kd)
    T_STATS[src] = query_top_stats(D)
    EXACT_TEST[src] = exact_pairs_test(src, te1, xe, CAND_CFG.get("exact_union", []))
    del xe
    free_memory()
    print(f"  RAM {ram_gb()}")
del te1
free_memory()
print(f"Test encoding + search: {(time.time()-t)/60:.1f} min")

# ---- 2) build candidate pairs + apply the pruner in query chunks (never holds all stage-1 pairs) ----
t = time.time()
top1_all_t = np.fmax(T_STATS[2][0], T_STATS[3][0])
unseen_q = ~np.isin(T1_CC, list(TRAIN_COUNTRY_CODES))
Q_CHUNK = 100_000
TEST_PAIRS, N_STAGE1_TEST = {}, 0
for src, df in ((2, test_s2), (3, test_s3)):
    I, D, kd = SEARCH[src]
    parts = {k: [] for k in ("q", "o", "sim", "rank", "via", "score")}
    n_before = 0
    for a in tqdm(range(0, tn1, Q_CHUNK), desc=f"S{src} candidates" + (" + pruning" if PRUNER is not None else "")):
        rows = np.arange(a, min(a + Q_CHUNK, tn1))
        m = keep_mask(I[rows], D[rows], kd, T1_CC[rows])
        qi, col = np.nonzero(m)
        q = rows[qi].astype(np.int32)
        o = I[rows[qi], col].astype(np.int32)
        sim = D[rows[qi], col].astype(np.float32)
        rank = col.astype(np.int32)
        n_before += len(q)
        if PRUNER is not None and len(q):
            score = score_pairs(PRUNER, test_s1, q, df, o, src, sim, rank.astype(np.float32),
                                T_STATS[src][0][q], T_STATS[src][1][q], top1_all_t[q])
            keep = score >= np.where(unseen_q[q], PRUNE_TAU_UNSEEN, PRUNE_TAU)
            q, o, sim, rank, score = q[keep], o[keep], sim[keep], rank[keep], score[keep]
        else:
            score = np.full(len(q), np.nan, np.float32)
        parts["q"].append(q); parts["o"].append(o); parts["sim"].append(sim); parts["rank"].append(rank)
        parts["via"].append(np.ones(len(q), np.int8)); parts["score"].append(score)
    P = {k: np.concatenate(v) for k, v in parts.items()}
    N_STAGE1_TEST += n_before
    if EXACT_TEST[src] is not None:                       # exact-key pairs bypass the pruner
        E = EXACT_TEST[src]
        P = {k: np.concatenate([P[k], E[k].astype(P[k].dtype)]) for k in P}
        P = dedupe_pairs(P, tn2 if src == 2 else tn3)
    P["src"] = src
    TEST_PAIRS[src] = P
    print(f"  S{src}: stage-1 {n_before:,} pairs -> final {len(P['q']):,} pairs")
    del SEARCH[src], I, D
    free_memory()
print(f"Candidate building{' + pruning' if PRUNER is not None else ''}: {(time.time()-t)/60:.1f} min | RAM {ram_gb()}")

## 28. Generate `candidate_pairs.tsv`
One row per test S1 entity. `candidate_entity_ids` holds S2/S3 test IDs sorted by decreasing cosine, with no
duplicates, and is empty when there are no candidates. This is exactly the set the next-stage matcher will score,
so every future match must be a subset of it.

`test_candidate_scores.parquet` holds the same pairs with `cosine`, `faiss_rank` (−1 = exact-key only),
`source`, provenance flags and the pruner's `prune_score`. These are free features for the matcher.

In [ ]:
q_all = np.concatenate([TEST_PAIRS[2]["q"], TEST_PAIRS[3]["q"]])
cand_code = np.concatenate([TEST_PAIRS[2]["o"], tn2 + TEST_PAIRS[3]["o"]])
sim_all = np.concatenate([TEST_PAIRS[2]["sim"], TEST_PAIRS[3]["sim"]])
rank_all = np.concatenate([TEST_PAIRS[2]["rank"], TEST_PAIRS[3]["rank"]])
via_all = np.concatenate([TEST_PAIRS[2]["via"], TEST_PAIRS[3]["via"]])
score_all = np.concatenate([TEST_PAIRS[s].get("score", np.full(len(TEST_PAIRS[s]["q"]), np.nan, np.float32))
                            for s in (2, 3)])
del TEST_PAIRS
order = np.lexsort((-sim_all, q_all))
q_all, cand_code, sim_all, rank_all, via_all, score_all = (a[order] for a in (q_all, cand_code, sim_all, rank_all,
                                                                             via_all, score_all))
del order

TEST_IDS = np.concatenate([test_s2["entity_id"].to_numpy(object), test_s3["entity_id"].to_numpy(object)])
S1_IDS = test_s1["entity_id"].to_numpy(object)
cand_ids = TEST_IDS[cand_code]
bounds = np.searchsorted(q_all, np.arange(tn1 + 1))
N_CANDS_PER_S1 = np.diff(bounds)

CANDIDATE_PATH = OUT_DIR / "candidate_pairs.tsv"
t = time.time()
with open(CANDIDATE_PATH, "w", encoding="utf-8", newline="") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    buf = []
    for i in range(tn1):
        buf.append(f"{S1_IDS[i]}\t{','.join(cand_ids[bounds[i]:bounds[i + 1]])}\n")
        if len(buf) >= 200_000:
            f.writelines(buf); buf = []
    f.writelines(buf)
print(f"Wrote {CANDIDATE_PATH} ({os.path.getsize(CANDIDATE_PATH)/1e6:,.0f} MB) in {time.time()-t:.0f}s")
print(f"Total candidate pairs: {len(q_all):,} | avg/S1 {N_CANDS_PER_S1.mean():.2f} | "
      f"empty rows {(N_CANDS_PER_S1 == 0).sum():,}")
by_country = pd.DataFrame({"country": test_s1["country"].values, "n": N_CANDS_PER_S1}).groupby("country")["n"]
print(by_country.agg(["count", "mean", "median", "max"]).to_string())

SCORES_PATH = None
if CONFIG["write_test_scores"]:
    import pyarrow as pa, pyarrow.parquet as pq
    SCORES_PATH = OUT_DIR / "test_candidate_scores.parquet"
    writer = None
    for a in range(0, len(q_all), 5_000_000):
        b = a + 5_000_000
        tbl = pa.table({
            "source1_entity_id": S1_IDS[q_all[a:b]],
            "candidate_entity_id": cand_ids[a:b],
            "source": np.where(cand_code[a:b] < tn2, "S2", "S3").astype(object),
            "cosine": sim_all[a:b],
            "faiss_rank": np.where(rank_all[a:b] == BIG_RANK, -1, rank_all[a:b]).astype(np.int32),
            "via_faiss": (via_all[a:b] & 1) > 0,
            "via_exact_full": (via_all[a:b] & 2) > 0,
            "via_exact_name": (via_all[a:b] & 4) > 0,
            "prune_score": score_all[a:b],
        })
        writer = writer or pq.ParquetWriter(SCORES_PATH, tbl.schema)
        writer.write_table(tbl)
    if writer:
        writer.close()
        print(f"Wrote {SCORES_PATH} ({os.path.getsize(SCORES_PATH)/1e6:,.0f} MB)")

## 29. Validate the candidate file
The file is re-read **from disk** and checked against every rule: exact header, one row per test S1 (none missing,
none duplicated, none extra), only S2-/S3- IDs that exist in the loaded test files (no S1 self-matches), and no
duplicate IDs within a row. Empty rows are empty strings, not `nan`. If `run_official_validator` is on, the official
`utils/validate_submission.py` row checker is also imported and run on this file. From the command line the
official tool also needs `matching_results.tsv`, which is the next stage's output.

In [ ]:
def validate_candidate_file(path, required_ids, valid_s2, valid_s3):
    errors = []
    def err(msg):
        if len(errors) < 50:
            errors.append(msg)
    seen, n_rows, n_ids = set(), 0, 0
    counts = {"dup_row": 0, "extra_row": 0, "dup_in_list": 0, "bad_prefix": 0, "unknown_id": 0, "bad_format": 0}
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        if header != ["source1_entity_id", "candidate_entity_ids"]:
            err(f"bad header {header}")
        for ln, line in enumerate(f, start=2):
            line = line.rstrip("\n")
            parts = line.split("\t")
            if len(parts) != 2:
                counts["bad_format"] += 1; err(f"line {ln}: expected 2 columns"); continue
            s1id, ids = parts
            n_rows += 1
            if s1id in seen:
                counts["dup_row"] += 1; err(f"duplicate row {s1id}")
            seen.add(s1id)
            if s1id not in required_ids:
                counts["extra_row"] += 1; err(f"unexpected S1 id {s1id}")
            if ids == "":
                continue
            if ids.strip().lower() == "nan":
                err(f"{s1id}: literal 'nan'")
            lst = ids.split(",")
            n_ids += len(lst)
            if len(set(lst)) != len(lst):
                counts["dup_in_list"] += 1; err(f"{s1id}: duplicate ids in list")
            for x in lst:
                if x.startswith("S2-"):
                    if x not in valid_s2: counts["unknown_id"] += 1; err(f"{s1id}: unknown {x}")
                elif x.startswith("S3-"):
                    if x not in valid_s3: counts["unknown_id"] += 1; err(f"{s1id}: unknown {x}")
                else:
                    counts["bad_prefix"] += 1; err(f"{s1id}: non S2/S3 id {x}")
    missing = required_ids - seen
    if missing:
        err(f"{len(missing):,} test S1 ids missing, e.g. {list(missing)[:3]}")
    ok = not errors
    print(f"rows={n_rows:,} ids={n_ids:,} missing={len(missing):,} " + " ".join(f"{k}={v}" for k, v in counts.items()))
    print("PASS - candidate_pairs.tsv satisfies all rules." if ok else f"FAIL - {len(errors)} issue(s):")
    for e in errors[:20]:
        print("  -", e)
    return ok

REQUIRED = set(S1_IDS.tolist())
IS_VALID = validate_candidate_file(CANDIDATE_PATH, REQUIRED, set(test_s2["entity_id"]), set(test_s3["entity_id"]))

if CONFIG["run_official_validator"] and VALIDATOR_PATH:
    spec = importlib.util.spec_from_file_location("validate_submission", VALIDATOR_PATH)
    vs = importlib.util.module_from_spec(spec); spec.loader.exec_module(vs)
    off_errors = []
    valid_ids = set(test_s2["entity_id"]) | set(test_s3["entity_id"])
    vs.validate_id_list_file(str(CANDIDATE_PATH), vs.CANDIDATE_HEADER, "candidate_entity_ids",
                             REQUIRED, valid_ids, off_errors)
    print("Official validator row checks:", "PASS" if not off_errors else off_errors[:10])
    IS_VALID = IS_VALID and not off_errors
    del valid_ids
if DEBUG_MODE:
    print("\nDEBUG_MODE: the file covers only the test subsample - it is NOT a submittable candidate file.")
else:
    print("\nCLI check once matching_results.tsv exists:\n  python3 utils/validate_submission.py "
          "--matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test")

## 30. Saved outputs

In [ ]:
OUTPUTS = {
    "recall_at_k.csv": OUT_DIR / "recall_at_k.csv",
    "recall_curve_per_source.csv": OUT_DIR / "recall_curve_per_source.csv",
    "similarity_threshold_sweep.csv": OUT_DIR / "similarity_threshold_sweep.csv",
    "baseline_comparison_mini.csv": OUT_DIR / "baseline_comparison_mini.csv",
    "validation_retrieval_details.csv": OUT_DIR / "validation_retrieval_details.csv",
    "val_candidate_pairs.parquet": OUT_DIR / "val_candidate_pairs.parquet",
    "val_queries.parquet": OUT_DIR / "val_queries.parquet",
    "candidate_config.json": OUT_DIR / "candidate_config.json",
    "pruner_tradeoff.csv": OUT_DIR / "pruner_tradeoff.csv",
    "pruner.pkl": OUT_DIR / "pruner.pkl",
    "candidate_pairs.tsv": CANDIDATE_PATH,
    "test_candidate_scores.parquet": SCORES_PATH,
    "embedding_model/": MODEL_DIR,
}
if PRETRAINED_FULL is not None:
    PRETRAINED_FULL.to_csv(OUT_DIR / "pretrained_vs_finetuned_full.csv", index=False)
    OUTPUTS["pretrained_vs_finetuned_full.csv"] = OUT_DIR / "pretrained_vs_finetuned_full.csv"
for name, p in OUTPUTS.items():
    if p is None or not Path(p).exists():
        print(f"  {name:36s} (not written)")
        continue
    size = sum(f.stat().st_size for f in Path(p).rglob("*")) if Path(p).is_dir() else Path(p).stat().st_size
    print(f"  {name:36s} {str(p):70s} {size/1e6:10,.1f} MB")

## 31. Summary / next steps

In [ ]:
fm = FINAL_VAL_METRICS
sweep_txt = SEL["sweep"][["k_s2", "pair_recall", "s2_recall", "s3_recall", "mean_s1_recall", "full_recall_pct",
                          "avg_cands"]].rename(columns={"k_s2": "K"}).to_string(index=False, float_format="{:.4f}".format)
mini_txt = (BASELINE_DF[BASELINE_DF.K == 10][["method", "pair_recall", "s2_recall", "s3_recall"]]
            .to_string(index=False, float_format="{:.4f}".format))
k_txt = (f"flat per-source: K_S2={CAND_CFG['k_s2']}, K_S3={CAND_CFG['k_s3']}" if CAND_CFG["mode"] == "flat" else
         f"dynamic: cosine >= {CAND_CFG['sim_floor']} (keep top {CAND_CFG['k_min']}, cap {CAND_CFG['dyn_k']}); "
         f"flat fallback K_S2={CAND_CFG['k_s2']}, K_S3={CAND_CFG['k_s3']}")
if PRUNE_METRICS is not None:
    pm, p1 = PRUNE_METRICS, PRUNE_STAGE1_METRICS
    prune_txt = (f"Candidate pruner (held-out validation half):\n"
                 f"  threshold {PRUNE_TAU:.4f} (unseen countries {PRUNE_TAU_UNSEEN:.4f})\n"
                 f"  candidates/S1 {p1['avg_cands']:.1f} -> {pm['avg_cands']:.2f} (p95 {pm['p95_cands']:.0f}), "
                 f"pair recall {p1['pair_recall']:.4f} -> {pm['pair_recall']:.4f}\n"
                 f"  singletons with empty candidate list: {pm['singleton_empty_pct']:.1f}%")
else:
    prune_txt = "Candidate pruner: disabled"
print(f"""
==================================================
CANDIDATE GENERATION SUMMARY {'(DEBUG MODE)' if DEBUG_MODE else ''}
==================================================

Model:
  {MODEL_NAME} ({N_PARAMS/1e6:.1f}M params, {CONFIG['model_license']}), mean-pool + L2, view {PRIMARY_VIEW}
  SupCon (tau={CONFIG['temperature']}), source-aware S1<->S2 / S1<->S3 batches,
  hard negatives: {HARD_NEGS is not None}, final encoder mode: {CONFIG['final_train_mode']}

Training entities:
  S1 {TRAIN_SUMMARY['train_s1']:,} | S2 {TRAIN_SUMMARY['train_s2']:,} | S3 {TRAIN_SUMMARY['train_s3']:,}
  entity groups: validation run {TRAIN_SUMMARY['train_groups']['validation_run']:,}, final {TRAIN_SUMMARY['train_groups']['final']:,}

Validation entities:
  {TRAIN_SUMMARY['val_s1']:,} S1 ({TRAIN_SUMMARY['val_singletons']:,} singletons), queried against the full S2/S3 indexes
  per-country partitioning: {PARTITION} (cross-country GT pairs {CROSS_COUNTRY_FRAC:.4%})

K results (same K for S2 and S3):
{sweep_txt}

Mini-benchmark Recall@10 (relative comparison):
{mini_txt}

Selected final K:
  {k_txt}
  exact-key union: {CAND_CFG.get('exact_union') or 'none'} | unseen countries: flat K x {CONFIG['unseen_country_k_multiplier']}
  validation: pair recall {fm['pair_recall']:.4f} (S2 {fm['s2_recall']:.4f}, S3 {fm['s3_recall']:.4f}),
              full-recall S1 {fm['full_recall_pct']:.2f}%, avg cands {fm['avg_cands']:.1f},
              singleton avg cands {fm['avg_cands_singleton']:.1f}, reduction ratio {fm['reduction_ratio']:.6f}
{prune_txt}

Test S1:
  {tn1:,}{'' if not DEBUG_MODE else f' (subsample of {N_TEST_S1_FULL:,})'}
Test S2:
  {tn2:,}
Test S3:
  {tn3:,}

Average candidates per S1:
  {N_CANDS_PER_S1.mean():.2f}  (total pairs {len(q_all):,}; before pruning {N_STAGE1_TEST:,})
  p95 {np.percentile(N_CANDS_PER_S1, 95):.0f} | empty rows {(N_CANDS_PER_S1 == 0).mean():.1%} | reduction ratio {1 - len(q_all) / (tn1 * (tn2 + tn3)):.8f}

candidate_pairs.tsv:
  {CANDIDATE_PATH}  | valid: {IS_VALID}

Total notebook runtime: {(time.time()-T_NOTEBOOK_START)/60:.1f} min
==================================================
""")

### Next stage (not implemented here)
```
candidate_pairs.tsv  (+ test_candidate_scores.parquet: cosine, rank, provenance)
        │
        ▼
pairwise feature extraction   name/address: Jaccard, Levenshtein/Jaro-Winkler, char-TF-IDF cosine, token overlap,
                              number/PIN/ZIP agreement, legal-suffix-stripped name equality, missing-field flags,
                              encoder cosine + rank, per-S1 context (rank among candidates, gap to top-1)
        │
        ▼
final matching model          e.g. GBDT on features, or a small MIT/Apache-2.0 cross-encoder (<= 8B params);
                              trained on the SAME train split, candidates generated by THIS blocker
        │
        ▼
F0.5-optimised threshold      tuned on validation with the per-S1 macro F0.5 (singletons included:
                              an empty prediction for a singleton scores 1.0)
        │
        ▼
matching_results.tsv          subset of candidate_pairs.tsv; validate with utils/validate_submission.py
```
**Tips for the matcher:** generate training pairs for the matcher by running this exact blocker on the validation
split (use `validation_retrieval_details.csv` and `candidate_config.json`), so the matcher learns the same negative
distribution it will see on test.